# Benchmark e análise das ferramentas de OCR/extração

O objetivo desta etapa é analisar e comparar o desempenho de diferentes ferramentas de extração textual aplicadas ao mesmo conjunto documental.

Neste experimento, serão avaliadas seis abordagens:

- PyMuPDF
- pdfplumber
- Tesseract
- EasyOCR
- Docling
- OCRMyPDF

A metodologia busca garantir uma comparação justa entre as ferramentas. Para isso, todas serão aplicadas sobre o mesmo PDF, mantendo padronizadas todas as etapas posteriores à extração textual.

## Fluxo metodológico

Mesmo PDF de entrada  
↓  
Ferramenta 1 — PyMuPDF  
Ferramenta 2 — pdfplumber  
Ferramenta 3 — Tesseract  
Ferramenta 4 — EasyOCR  
Ferramenta 5 — Docling  
Ferramenta 6 — OCRMyPDF  
↓  
Mesmo REGEX_CAMPOS  
↓  
Mesmo extrator de campos  
↓  
Mesmo processo de pós-processamento, quando aplicável  
↓  
Mesmo avaliador  
↓  
Comparação final dos resultados  

Dessa forma, a principal variável alterada entre os testes será a ferramenta responsável pela extração textual inicial. Todas as demais etapas permanecerão padronizadas, permitindo identificar com maior precisão o impacto de cada ferramenta na qualidade dos dados extraídos.

## Campos avaliados

A comparação final considerará os resultados obtidos em cada campo extraído:

- nome_escola
- codigo_inep
- nome_gre
- produto
- total_kg
- data_inicio_obs
- data_fim_obs

## Resultados analisados

Também serão analisadas duas versões dos resultados:

### Resultado bruto

OCR/extração + regex

### Resultado pós-processado

OCR/extração + regex + substituições + base auxiliar de escolas por GRE

Com isso, será possível comparar não apenas o desempenho original de cada ferramenta, mas também o ganho obtido após a etapa de pós-processamento. Essa separação é importante para diferenciar a capacidade da ferramenta de extração textual da contribuição das regras de correção e padronização aplicadas posteriormente.

## A única coisa que muda entre os Colabs é o método de extração do texto. Todo o resto deve ser igual.

Com a seguinte estrutura:

---

### 🏭 BLOCO 1 — PIPELINE DE PRODUÇÃO

PASSO 1 — Instalar e importar bibliotecas  
PASSO 2 — Upload dos arquivos de entrada (PDF + base de escolas)  
PASSO 3 — Extrair texto por página com a ferramenta escolhida  
PASSO 4 — Definir CAMPOS_SAIDA e REGEX_CAMPOS  
PASSO 5 — Definir funções de normalização e pós-processamento  
PASSO 6 — Aplicar função de extração dos dados  
PASSO 7 — Salvar resultado bruto da ferramenta  
PASSO 8 — Aplicar pós-processamento e salvar novo resultado  
PASSO 9 — Gerar relatório de correções (Excel + PDF)

---

### 🔬 BLOCO 2 — BENCHMARK *(somente desenvolvimento)*
> ⚠️ Este bloco não estará presente em produção.

PASSO 10 — Upload do gabarito  
PASSO 11 — Avaliar resultado pós-processado contra gabarito  
PASSO 12 — Gerar relatório de métricas

# PASSO 1 — Instalar e importar bibliotecas

In [ ]:
# Tesseract
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por

# Python libs
!pip install -q pytesseract pymupdf opencv-python-headless jiwer editdistance

import re, os
import json
import time
import fitz
import numpy as np
import cv2
import pytesseract

from PIL import Image
from datetime import datetime
from google.colab import files

# ─────────────────────────────────────────────
# IDENTIFICADOR DA FERRAMENTA
# Altere aqui ao copiar o notebook para outra ferramenta.
# ─────────────────────────────────────────────
NOME_FERRAMENTA = "pymupdf_tesseract"

print("Bibliotecas carregadas.")
print("Ferramenta:", NOME_FERRAMENTA)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package tesseract-ocr-por.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-por_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-por (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-por (1:4.00~git30-7274cfa-1.1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 103.2 MB/s eta 0:00:00
Bibliotecas carregadas.
Ferramenta: pymupdf_tesseract


# PASSO 2 — Upload dos arquivos de entrada (PDF + base de escolas)

In [ ]:
uploaded = files.upload()

pdf_path = next((k for k in uploaded.keys() if k.lower().endswith(".pdf")), None)

if pdf_path is None:
    raise ValueError("Nenhum arquivo PDF encontrado. Envie apenas o PDF neste passo.")

print("Arquivo PDF carregado:", pdf_path)

Saving GREsUnificado.pdf to GREsUnificado.pdf
Arquivo PDF carregado: GREsUnificado.pdf


In [ ]:
# ============================================================
# PASSO 2.1 — UPLOAD E LIMPEZA DA BASE DE ESCOLAS POR GRE
# ============================================================

from google.colab import files
import pandas as pd
import re
import unicodedata

print("Envie o arquivo CSV da base de escolas por GRE.")
print("A base pode ter várias colunas, mas serão usadas apenas:")
print("- nome_gre")
print("- codigo_inep")
print("- nome_escola")

uploaded_base = files.upload()

arquivo_base_escolas = None

for nome_arquivo in uploaded_base.keys():
    if nome_arquivo.lower().endswith(".csv"):
        arquivo_base_escolas = nome_arquivo
        break

if arquivo_base_escolas is None:
    raise ValueError("Nenhum arquivo CSV foi enviado para a base de escolas.")

base_original = pd.read_csv(arquivo_base_escolas, dtype=str).fillna("")

def normalizar_nome_coluna(coluna):
    coluna = str(coluna).strip().lower()
    coluna = unicodedata.normalize("NFD", coluna)
    coluna = "".join(c for c in coluna if unicodedata.category(c) != "Mn")
    coluna = re.sub(r"[^a-z0-9]+", "_", coluna)
    coluna = re.sub(r"_+", "_", coluna).strip("_")
    return coluna

base_original.columns = [normalizar_nome_coluna(c) for c in base_original.columns]

possiveis_colunas = {
    "nome_gre": [
        "nome_gre",
        "gre",
        "regional",
        "gerencia",
        "gerencia_regional",
        "gerencia_regional_de_educacao"
    ],
    "codigo_inep": [
        "codigo_inep",
        "inep",
        "cod_inep",
        "codigo",
        "codigo_escola",
        "cod_escola"
    ],
    "nome_escola": [
        "nome_escola",
        "escola",
        "nome",
        "unidade",
        "unidade_escolar",
        "nome_da_escola"
    ]
}

colunas_encontradas = {}

for coluna_padrao, alternativas in possiveis_colunas.items():
    encontrada = None

    for alternativa in alternativas:
        if alternativa in base_original.columns:
            encontrada = alternativa
            break

    if encontrada is None:
        raise ValueError(
            f"Não foi possível encontrar a coluna obrigatória '{coluna_padrao}'. "
            f"Colunas disponíveis na base: {list(base_original.columns)}"
        )

    colunas_encontradas[coluna_padrao] = encontrada

base_escolas = base_original[
    [
        colunas_encontradas["nome_gre"],
        colunas_encontradas["codigo_inep"],
        colunas_encontradas["nome_escola"]
    ]
].copy()

base_escolas.columns = ["nome_gre", "codigo_inep", "nome_escola"]

base_escolas["nome_gre"] = base_escolas["nome_gre"].fillna("").astype(str).str.strip()
base_escolas["codigo_inep"] = base_escolas["codigo_inep"].fillna("").astype(str).str.strip()
base_escolas["nome_escola"] = base_escolas["nome_escola"].fillna("").astype(str).str.strip()

base_escolas = base_escolas[
    (base_escolas["nome_gre"] != "") |
    (base_escolas["codigo_inep"] != "") |
    (base_escolas["nome_escola"] != "")
].copy()

base_escolas = base_escolas.drop_duplicates(
    subset=["nome_gre", "codigo_inep", "nome_escola"]
).reset_index(drop=True)



Envie o arquivo CSV da base de escolas por GRE.
A base pode ter várias colunas, mas serão usadas apenas:
- nome_gre
- codigo_inep
- nome_escola


Saving base_escolas_gres_ceasa.csv to base_escolas_gres_ceasa.csv


# PASSO 3 — Extrair texto por página com a ferramenta escolhida

In [ ]:
def pagina_para_imagem(page, dpi=300):
    zoom = dpi / 72
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)

    img_np = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
        pix.height, pix.width, pix.n
    )

    img_cv2 = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    return img_cv2


def ocr_tesseract_pagina(img_cv2):
    return pytesseract.image_to_string(
        img_cv2,
        config="--oem 3 --psm 6 -l por"
    )


def cv2_para_pil(img_cv2):
    img_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img_rgb)


def extrair_textos_pymupdf_ocr(pdf_path):
    doc = fitz.open(pdf_path)
    paginas = []

    for i, page in enumerate(doc):
        img_cv2 = pagina_para_imagem(page, dpi=300)
        texto = ocr_tesseract_pagina(img_cv2)
        img_pil = cv2_para_pil(img_cv2)

        paginas.append({
            "pagina": i + 1,
            "texto": texto,
            "imagem": img_pil
        })

        print(f"Página {i + 1} processada")

    return paginas

# PASSO 4 — Definir CAMPOS_SAIDA e REGEX_CAMPOS

In [ ]:
CAMPOS_SAIDA = [
    "pagina",
    "nome_escola",
    "codigo_inep",
    "nome_gre",
    "produto",
    "total_kg",
    "data_inicio_obs",
    "data_fim_obs"
]

In [ ]:
# regex padrão para documentos digitados para todes os outros.

REGEX_CEASA = {
    "codigo_inep": [
        r"ESCOLA\s*:\s*\n?\s*([0-9]{8}[A-Z]?)\s*[-–]",
        r"\bINEP\s*[:\-]?\s*N?[ºo]?\s*([0-9]{8})",
        r"C[oó]digo\s+INEP\s*[:\-]?\s*([0-9]{8})"
    ],

    "nome_escola": [
        r"ESCOLA\s*:\s*\n?\s*[0-9]{8}[A-Z]?\s*[-–]\s*(.*?)(?=\n\s*(?:AV|AV\.|AVENIDA|RUA|R\b|R\.|EST|ESTRADA|ENG|PR|PRAÇA|PRACA|SITIO|SÍTIO|ROD|RODOVIA|TRAVESSA)\b|\n\s*(?:GRE|nRF|GFE|GRF)\s*[:•]|\n\s*Fornecimento\s+do\s+produto)",
        r"ESCOLA\s*:\s*(.*?)(?=\n\s*(?:AV|AV\.|AVENIDA|RUA|R\b|R\.|EST|ESTRADA|ENG|PR|PRAÇA|PRACA|SITIO|SÍTIO|ROD|RODOVIA|TRAVESSA)\b|\n\s*(?:GRE|nRF|GFE|GRF)\s*[:•]|\n\s*Fornecimento\s+do\s+produto)"
    ],

    "nome_gre": [
        r"(?:GRE|nRF|GFE|GRF)\s*[:•]?\s*(?:GRE\s+)?(.*?)(?=\n\s*Fornecimento\s+do\s+produto)",
        r"(?:GRE|nRF|GFE|GRF)\s*[:•]?\s*(?:GRE\s+)?([A-ZÁÀÃÂÉÊÍÓÔÕÚÇ\s]+?)(?=\n|Fornecimento|$)"
    ],

    "produto": [
        r"Fornecimento\s+do\s+produto\s*:\s*(.*?)(?=\n\s*Medida\s+de\s+Fornecimento)",
        r"produto\s*:\s*(.*?)(?=\n\s*Medida\s+de\s+Fornecimento|\n|$)"
    ],

    "total_kg": [
        r"\bTotal\s*:\s*([0-9]+(?:[,.][0-9]+)?)\s*KG\b",
        r"\bTotal\s*:\s*([0-9]+(?:[,.][0-9]+)?)"
    ],

    "data_inicio_obs": [
        r"per[ií]odo\s+de\s+distribui[çc][aã]o.*?de\s*([0-3]?\d/[01]?\d)\s*[àa]\s*[0-3]?\d/[01]?\d"
    ],

    "data_fim_obs": [
        r"per[ií]odo\s+de\s+distribui[çc][aã]o.*?de\s*[0-3]?\d/[01]?\d\s*[àa]\s*([0-3]?\d/[01]?\d)"
    ]
}

# PASSO 5 — Definir funções de normalização e pós-processamento

In [ ]:
# ============================================================
# PASSO 5 — FUNÇÕES DE NORMALIZAÇÃO, BUSCA E PÓS-PROCESSAMENTO
# ============================================================

import re
import unicodedata
import pandas as pd
from difflib import SequenceMatcher

# ------------------------------------------------------------
# DICIONÁRIO DE SUBSTITUIÇÕES PARA GRE
# ------------------------------------------------------------

SUBSTITUICOES_GRE = {

    # ── METRO NORTE ──────────────────────────────────────────
    "MFTRO NORTE":            "METRO NORTE",
    "MFTRO NORTF":            "METRO NORTE",
    "METRO NORTF":            "METRO NORTE",
    "M3TRO NORTE":            "METRO NORTE",
    "M3TRO NORTF":            "METRO NORTE",
    "METROPOLITANA NORTE":    "METRO NORTE",
    "MFTRO":                  "METRO",
    "NORTF":                  "NORTE",
    "N0RTE":                  "NORTE",
    "N0RTF":                  "NORTE",

    # ── METRO SUL ────────────────────────────────────────────
    "MFTRO SUL":              "METRO SUL",
    "METRO 5UL":              "METRO SUL",
    "MFTRO 5UL":              "METRO SUL",
    "M3TRO SUL":              "METRO SUL",
    "M3TRO 5UL":              "METRO SUL",
    "METRO SVL":              "METRO SUL",
    "MFTRO SVL":              "METRO SUL",

    # ── RECIFE NORTE ─────────────────────────────────────────
    "RFCIFE NORTE":           "RECIFE NORTE",
    "RFCIFE NORTF":           "RECIFE NORTE",
    "RECIFE NORTF":           "RECIFE NORTE",
    "REC1FE NORTE":           "RECIFE NORTE",
    "RECIFE N0RTE":           "RECIFE NORTE",
    "REC1FE N0RTE":           "RECIFE NORTE",
    "RFCIFE N0RTE":           "RECIFE NORTE",

    # ── RECIFE SUL ───────────────────────────────────────────
    "RECIFE SUI":             "RECIFE SUL",
    "RECIFE 5UL":             "RECIFE SUL",
    "RFCIFE SUL":             "RECIFE SUL",
    "RFCIFE 5UL":             "RECIFE SUL",
    "REC1FE SUL":             "RECIFE SUL",
    "REC1FE 5UL":             "RECIFE SUL",
    "RECIFE SUL":             "RECIFE SUL",

    # ── CARUARU ──────────────────────────────────────────────
    "CARUAFU":                "CARUARU",
    "GARUARU":                "CARUARU",
    "CARUAKU":                "CARUARU",
    "C4RUARU":                "CARUARU",
    "CARV4RU":                "CARUARU",
    "CARU4RU":                "CARUARU",

    # ── GARANHUNS ────────────────────────────────────────────
    "GARANHUN5":              "GARANHUNS",
    "6ARANHUNS":              "GARANHUNS",
    "6ARANHUN5":              "GARANHUNS",
    "GARANHU N5":             "GARANHUNS",
    "GARANHU NS":             "GARANHUNS",
    "GARANHVNS":              "GARANHUNS",
    "GARANHU5":               "GARANHUNS",

    # ── VITÓRIA DE STO ANTÃO ─────────────────────────────────
    "VI70RIA DE STO ANTAO":   "VITORIA DE STO ANTAO",
    "V1TORIA DE STO ANTAO":   "VITORIA DE STO ANTAO",
    "VITORIA DE ST0 ANTAO":   "VITORIA DE STO ANTAO",
    "VITORIA DE STO ANT4O":   "VITORIA DE STO ANTAO",
    "VI70RIA DE ST0 ANTAO":   "VITORIA DE STO ANTAO",
    "VITORIA DE STO ANT AO":  "VITORIA DE STO ANTAO",

    # ── NAZARÉ DA MATA ───────────────────────────────────────
    "NA2ARE DA MATA":         "NAZARE DA MATA",
    "MAZARE DA MATA":         "NAZARE DA MATA",
    "NA2ARE DA MAT4":         "NAZARE DA MATA",
    "NAZA RE DA MATA":        "NAZARE DA MATA",

    # ── PALMARES ─────────────────────────────────────────────
    "P4LMARES":               "PALMARES",
    "PAIMARES":               "PALMARES",
    "PALMA RES":              "PALMARES",
    "PALMA RE5":              "PALMARES",
    "P4LMARE5":               "PALMARES",

    # ── AFOGADOS DA INGAZEIRA ────────────────────────────────
    "AF0GADOS DA INGAZEIRA":  "AFOGADOS DA INGAZEIRA",
    "AFOGAD0S DA INGAZEIRA":  "AFOGADOS DA INGAZEIRA",
    "AF0GAD0S DA INGAZEIRA":  "AFOGADOS DA INGAZEIRA",
    "AFOGADOS DA 1NGAZEIRA":  "AFOGADOS DA INGAZEIRA",
    "AFOGADOS DA INGAZE1RA":  "AFOGADOS DA INGAZEIRA",
    "AFOGADOS DA INGA2EIRA":  "AFOGADOS DA INGAZEIRA",
    "AF0GADOS DA 1NGAZEIRA":  "AFOGADOS DA INGAZEIRA",

    # ── SALGUEIRO ────────────────────────────────────────────
    "5ALGUEIRO":              "SALGUEIRO",
    "SALGU3IRO":              "SALGUEIRO",
    "5ALGU3IRO":              "SALGUEIRO",
    "SALGU EIRO":             "SALGUEIRO",
    "5ALGU EIRO":             "SALGUEIRO",

    # ── ARARIPINA ────────────────────────────────────────────
    "4RARIPINA":              "ARARIPINA",
    "ARARIPIMA":              "ARARIPINA",
    "ARARIP1NA":              "ARARIPINA",
    "4RARIPIMA":              "ARARIPINA",
    "ARARIP1MA":              "ARARIPINA",

    # ── ARCOVERDE ────────────────────────────────────────────
    "ARC0VERDE":              "ARCOVERDE",
    "4RCOVERDE":              "ARCOVERDE",
    "ARCOVERDF":              "ARCOVERDE",
    "ARC0VERDF":              "ARCOVERDE",
    "4RC0VERDE":              "ARCOVERDE",

    # ── PETROLINA ────────────────────────────────────────────
    "P3TROLINA":              "PETROLINA",
    "PETROIINA":              "PETROLINA",
    "PET R0LINA":             "PETROLINA",
    "P3TR0LINA":              "PETROLINA",
    "PETR0LINA":              "PETROLINA",

    # ── FLORESTA ─────────────────────────────────────────────
    "FL0RESTA":               "FLORESTA",
    "FL0RE5TA":               "FLORESTA",
    "FLOFE5TA":               "FLORESTA",
    "FLOIESTA":               "FLORESTA",

    # ── LIMOEIRO ─────────────────────────────────────────────
    "L1MOEIRO":               "LIMOEIRO",
    "LIM0EIRO":               "LIMOEIRO",
    "1IMOEIRO":               "LIMOEIRO",
    "LIMOE1RO":               "LIMOEIRO",
    "L1M0EIRO":               "LIMOEIRO",
}

# ------------------------------------------------------------
# DICIONÁRIO DE SUBSTITUIÇÕES PARA PRODUTOS / ALIMENTOS
# ------------------------------------------------------------

SUBSTITUICOES_PRODUTOS = {
    "FRANGO COXA E SOBRE COXA": "FRANGO COXA E SOBRECOXA",
    "FRANGO COXA E SOBRECOXA": "FRANGO COXA E SOBRECOXA",
    "FRANGO COXA E S0BRECOXA": "FRANGO COXA E SOBRECOXA",
    "FRANGO COXA E SOBRFCOXA": "FRANGO COXA E SOBRECOXA",
    "FRANGO COXA E S0BRFCOXA": "FRANGO COXA E SOBRECOXA",
    "FRANGO COXA E SOBRECOX4": "FRANGO COXA E SOBRECOXA",
    "FRANGO COXA E SOBRECOX A": "FRANGO COXA E SOBRECOXA",
    "FRANGO COXA E SOBRECOXA O": "FRANGO COXA E SOBRECOXA",
    "FRANGO COXA E SOBRECOXA 0": "FRANGO COXA E SOBRECOXA"
}

# ------------------------------------------------------------
# FUNÇÕES GERAIS DE NORMALIZAÇÃO
# ------------------------------------------------------------

def remover_acentos(texto):
    if texto is None:
        return ""

    texto = str(texto)
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")

    return texto


def normalizar_texto(texto):
    if texto is None:
        return ""

    texto = str(texto).upper().strip()
    texto = remover_acentos(texto)

    texto = texto.replace("?", " ")
    texto = texto.replace("º", "")
    texto = texto.replace("ª", "")
    texto = texto.replace("°", "")

    texto = re.sub(r"[^A-Z0-9\s\-]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


def normalizar_codigo_inep(codigo):
    if codigo is None:
        return ""

    codigo = str(codigo).upper().strip()
    codigo = remover_acentos(codigo)
    codigo = re.sub(r"[^0-9A-Z]", "", codigo)

    return codigo


def aplicar_substituicoes(texto, substituicoes):
    texto_norm = normalizar_texto(texto)

    for errado, correto in substituicoes.items():
        errado_norm = normalizar_texto(errado)
        correto_norm = normalizar_texto(correto)

        if errado_norm in texto_norm:
            texto_norm = texto_norm.replace(errado_norm, correto_norm)

    texto_norm = re.sub(r"\s+", " ", texto_norm).strip()

    return texto_norm



# ------------------------------------------------------------
# MAPA DE GREs CANÔNICAS  (fallback pós-substituição)
# Ordene do mais específico para o mais genérico.
# ------------------------------------------------------------

GRES_CANONICAS = [
    ("AFOGADOS DA INGAZEIRA", "AFOGADOS DA INGAZEIRA"),
    ("VITORIA DE STO ANTAO",  "VITORIA DE STO ANTAO"),
    ("NAZARE DA MATA",        "NAZARE DA MATA"),
    ("METRO NORTE",           "METRO NORTE"),
    ("METRO SUL",             "METRO SUL"),
    ("RECIFE NORTE",          "RECIFE NORTE"),
    ("RECIFE SUL",            "RECIFE SUL"),
    ("CARUARU",               "CARUARU"),
    ("GARANHUNS",             "GARANHUNS"),
    ("PALMARES",              "PALMARES"),
    ("SALGUEIRO",             "SALGUEIRO"),
    ("ARARIPINA",             "ARARIPINA"),
    ("ARCOVERDE",             "ARCOVERDE"),
    ("PETROLINA",             "PETROLINA"),
    ("FLORESTA",              "FLORESTA"),
    ("LIMOEIRO",              "LIMOEIRO"),
]

# ------------------------------------------------------------
# NORMALIZAÇÃO ESPECÍFICA DE GRE
# ------------------------------------------------------------

def normalizar_gre(nome_gre):
    gre = normalizar_texto(nome_gre)

    gre = gre.replace("GRE:", " ")
    gre = gre.replace("G R E", " ")
    gre = gre.replace("NRE", " ")
    gre = gre.replace("NRF", " ")
    gre = gre.replace("ORF", " ")
    gre = gre.replace("DRF", " ")
    gre = gre.replace("RRF", " ")

    gre = re.sub(r"\bGRE\b", " ", gre)
    gre = re.sub(r"\s+", " ", gre).strip()

    gre = aplicar_substituicoes(gre, SUBSTITUICOES_GRE)

    # Fallback: colapsa para GRE canônica se substring conhecida detectada
    for chave, canonical in GRES_CANONICAS:
        if chave in gre:
            gre = canonical
            break

    if gre:
        gre = "GRE " + gre
    else:
        gre = ""

    gre = re.sub(r"\s+", " ", gre).strip()

    return gre


# ------------------------------------------------------------
# NORMALIZAÇÃO ESPECÍFICA DE PRODUTO
# ------------------------------------------------------------

def normalizar_produto(produto):
    produto = normalizar_texto(produto)
    produto = aplicar_substituicoes(produto, SUBSTITUICOES_PRODUTOS)

    if "FRANGO COXA" in produto and "SOBRECOXA" in produto:
        produto = "FRANGO COXA E SOBRECOXA"

    produto = re.sub(r"\s+", " ", produto).strip()

    return produto


# ------------------------------------------------------------
# SIMILARIDADE ENTRE NOMES
# ------------------------------------------------------------

def calcular_similaridade(a, b):
    a = normalizar_texto(a)
    b = normalizar_texto(b)

    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio()


# ------------------------------------------------------------
# PREPARAÇÃO DA BASE DE ESCOLAS
# ------------------------------------------------------------

def preparar_base_escolas(base_escolas):
    base = base_escolas.copy()

    colunas_obrigatorias = ["nome_gre", "codigo_inep", "nome_escola"]

    for coluna in colunas_obrigatorias:
        if coluna not in base.columns:
            raise ValueError(f"A coluna obrigatória '{coluna}' não existe na base.")

    base = base[colunas_obrigatorias].copy()

    base["nome_gre"] = base["nome_gre"].fillna("").astype(str).str.strip()
    base["codigo_inep"] = base["codigo_inep"].fillna("").astype(str).str.strip()
    base["nome_escola"] = base["nome_escola"].fillna("").astype(str).str.strip()

    base["nome_gre_norm"] = base["nome_gre"].apply(normalizar_gre)
    base["codigo_inep_norm"] = base["codigo_inep"].apply(normalizar_codigo_inep)
    base["nome_escola_norm"] = base["nome_escola"].apply(normalizar_texto)

    base = base.drop_duplicates(
        subset=["nome_gre_norm", "codigo_inep_norm", "nome_escola_norm"]
    ).reset_index(drop=True)

    return base


# ------------------------------------------------------------
# BUSCA DA ESCOLA NA BASE
# ------------------------------------------------------------

def buscar_escola_na_base(codigo_inep, nome_gre, nome_escola_extraido, base_escolas):
    codigo_norm = normalizar_codigo_inep(codigo_inep)
    gre_norm = normalizar_gre(nome_gre)
    escola_norm = normalizar_texto(nome_escola_extraido)

    # --------------------------------------------------------
    # 1. Busca principal: código INEP + GRE
    # --------------------------------------------------------

    if codigo_norm and gre_norm:
        busca = base_escolas[
            (base_escolas["codigo_inep_norm"] == codigo_norm) &
            (base_escolas["nome_gre_norm"] == gre_norm)
        ]

        if not busca.empty:
            linha = busca.iloc[0]
            return {
                "nome_gre": linha["nome_gre"],
                "codigo_inep": linha["codigo_inep"],
                "nome_escola": linha["nome_escola"],
                "metodo": "codigo_inep+gre",
                "similaridade": 1.0
            }

    # --------------------------------------------------------
    # 2. Busca alternativa: somente código INEP
    # --------------------------------------------------------

    if codigo_norm:
        busca = base_escolas[
            base_escolas["codigo_inep_norm"] == codigo_norm
        ]

        if not busca.empty:
            linha = busca.iloc[0]
            return {
                "nome_gre": linha["nome_gre"],
                "codigo_inep": linha["codigo_inep"],
                "nome_escola": linha["nome_escola"],
                "metodo": "codigo_inep",
                "similaridade": 1.0
            }

    # --------------------------------------------------------
    # 3. Busca por similaridade dentro da GRE
    #    Usada quando o INEP veio vazio ou não foi encontrado.
    # --------------------------------------------------------

    if escola_norm and gre_norm:
        candidatos = base_escolas[
            base_escolas["nome_gre_norm"] == gre_norm
        ].copy()

        if not candidatos.empty:
            candidatos["similaridade"] = candidatos["nome_escola_norm"].apply(
                lambda nome_base: calcular_similaridade(escola_norm, nome_base)
            )

            candidatos = candidatos.sort_values(
                by="similaridade",
                ascending=False
            )

            melhor = candidatos.iloc[0]

            if float(melhor["similaridade"]) >= 0.82:
                codigo_base = str(melhor["codigo_inep"]).strip()

                return {
                    "nome_gre": melhor["nome_gre"],
                    "codigo_inep": codigo_base if codigo_base else codigo_norm,
                    "nome_escola": melhor["nome_escola"],
                    "metodo": "similaridade_nome+gre",
                    "similaridade": float(melhor["similaridade"])
                }

    # --------------------------------------------------------
    # 4. Busca por similaridade em toda a base
    #    Último recurso, com limiar mais alto.
    # --------------------------------------------------------

    if escola_norm:
        candidatos = base_escolas.copy()

        candidatos["similaridade"] = candidatos["nome_escola_norm"].apply(
            lambda nome_base: calcular_similaridade(escola_norm, nome_base)
        )

        candidatos = candidatos.sort_values(
            by="similaridade",
            ascending=False
        )

        melhor = candidatos.iloc[0]

        if float(melhor["similaridade"]) >= 0.90:
            codigo_base = str(melhor["codigo_inep"]).strip()

            return {
                "nome_gre": melhor["nome_gre"],
                "codigo_inep": codigo_base if codigo_base else codigo_norm,
                "nome_escola": melhor["nome_escola"],
                "metodo": "similaridade_nome",
                "similaridade": float(melhor["similaridade"])
            }

    return None


# ------------------------------------------------------------
# PÓS-PROCESSAMENTO DE UM REGISTRO
# ------------------------------------------------------------

def pos_processar_registro(registro, base_escolas):
    registro_corrigido = dict(registro)

    auditoria = {
        "base_encontrada": False,
        "gre_corrigida": False,
        "produto_corrigido": False,
        "escola_corrigida_por_base": False,
        "codigo_inep_corrigido_por_base": False,
        "gre_corrigida_por_base": False,
        "metodo_escola": "",
        "similaridade_escola": "",
        "valores_originais": {}
    }

    # --------------------------------------------------------
    # GRE — normalização inicial
    # --------------------------------------------------------

    gre_original = registro_corrigido.get("nome_gre", "")
    gre_corrigida = normalizar_gre(gre_original)

    if normalizar_texto(gre_original) != normalizar_texto(gre_corrigida):
        auditoria["gre_corrigida"] = True
        auditoria["valores_originais"]["nome_gre"] = gre_original

    registro_corrigido["nome_gre"] = gre_corrigida

    # --------------------------------------------------------
    # PRODUTO
    # --------------------------------------------------------

    produto_original = registro_corrigido.get("produto", "")
    produto_corrigido = normalizar_produto(produto_original)

    if normalizar_texto(produto_original) != normalizar_texto(produto_corrigido):
        auditoria["produto_corrigido"] = True
        auditoria["valores_originais"]["produto"] = produto_original

    registro_corrigido["produto"] = produto_corrigido

    # --------------------------------------------------------
    # CÓDIGO INEP — normalização inicial
    # --------------------------------------------------------

    codigo_original = registro_corrigido.get("codigo_inep", "")
    codigo_corrigido = normalizar_codigo_inep(codigo_original)

    registro_corrigido["codigo_inep"] = codigo_corrigido

    # --------------------------------------------------------
    # BUSCA DA ESCOLA NA BASE
    # --------------------------------------------------------

    escola_encontrada = buscar_escola_na_base(
        codigo_inep=registro_corrigido.get("codigo_inep", ""),
        nome_gre=registro_corrigido.get("nome_gre", ""),
        nome_escola_extraido=registro_corrigido.get("nome_escola", ""),
        base_escolas=base_escolas
    )

    if escola_encontrada:
        auditoria["base_encontrada"] = True
        auditoria["metodo_escola"] = escola_encontrada.get("metodo", "")
        auditoria["similaridade_escola"] = escola_encontrada.get("similaridade", "")

        nome_escola_antes = registro_corrigido.get("nome_escola", "")
        codigo_inep_antes = registro_corrigido.get("codigo_inep", "")
        nome_gre_antes = registro_corrigido.get("nome_gre", "")

        nome_escola_base = escola_encontrada.get("nome_escola", "")
        codigo_inep_base = escola_encontrada.get("codigo_inep", "")
        nome_gre_base = escola_encontrada.get("nome_gre", "")

        # ----------------------------------------------------
        # CORRIGE NOME DA ESCOLA SOMENTE SE FOR DIFERENTE
        # ----------------------------------------------------

        if nome_escola_base and normalizar_texto(nome_escola_antes) != normalizar_texto(nome_escola_base):
            auditoria["escola_corrigida_por_base"] = True
            auditoria["valores_originais"]["nome_escola"] = nome_escola_antes
            registro_corrigido["nome_escola"] = nome_escola_base

        # ----------------------------------------------------
        # CORRIGE CÓDIGO INEP SOMENTE SE FOR DIFERENTE
        # ----------------------------------------------------

        if codigo_inep_base and normalizar_codigo_inep(codigo_inep_antes) != normalizar_codigo_inep(codigo_inep_base):
            auditoria["codigo_inep_corrigido_por_base"] = True
            auditoria["valores_originais"]["codigo_inep"] = codigo_inep_antes
            registro_corrigido["codigo_inep"] = codigo_inep_base

        # ----------------------------------------------------
        # CORRIGE GRE SOMENTE SE FOR DIFERENTE
        # ----------------------------------------------------

        if nome_gre_base and normalizar_gre(nome_gre_antes) != normalizar_gre(nome_gre_base):
            auditoria["gre_corrigida_por_base"] = True
            auditoria["valores_originais"]["nome_gre_antes_base"] = nome_gre_antes
            registro_corrigido["nome_gre"] = nome_gre_base

    registro_corrigido["_pos_processamento"] = auditoria

    return registro_corrigido


# ------------------------------------------------------------
# PREPARA A BASE
# ------------------------------------------------------------

if "base_escolas" not in globals():
    raise NameError("A variável 'base_escolas' não foi encontrada. Execute o Passo 2.1 antes deste passo.")

base_escolas = preparar_base_escolas(base_escolas)

print("Funções carregadas com sucesso.")
print("Base de escolas preparada.")
print("Total de registros na base:", len(base_escolas))

print("\nQuantidade de registros com INEP preenchido:")
print((base_escolas["codigo_inep_norm"] != "").sum())

print("\nQuantidade de registros sem INEP:")
print((base_escolas["codigo_inep_norm"] == "").sum())

print("\nPrévia da base preparada:")
display(base_escolas[["nome_gre", "codigo_inep", "nome_escola"]].head(20))

Funções carregadas com sucesso.
Base de escolas preparada.
Total de registros na base: 884

Quantidade de registros com INEP preenchido:
884

Quantidade de registros sem INEP:
0

Prévia da base preparada:


,nome_gre,codigo_inep,nome_escola
0,GRE CARUARU,26053756,CENTRO DE REABILITAÇÃO EDUCAÇÃO ESPECIAL ROTAR...
1,GRE CARUARU,26054019,ESCOLA DE REFERÊNCIA EM ENSINO FUNDAMENTAL E M...
2,GRE CARUARU,26062836,ESCOLA DE REFERÊNCIA EM ENSINO FUNDAMENTAL E M...
3,GRE CARUARU,26052792,ESCOLA DE REFERÊNCIA EM ENSINO FUNDAMENTAL E M...
4,GRE CARUARU,26064030,ESCOLA DE REFERÊNCIA EM ENSINO FUNDAMENTAL MAL...
5,GRE CARUARU,26050196,ESCOLA DE REFERÊNCIA EM ENSINO FUNDAMENTAL PRO...
6,GRE CARUARU,26054949,ESCOLA DE REFERÊNCIA EM ENSINO FUNDAMENTAL PRO...
7,GRE CARUARU,26052083,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO ANDRÉ COR...
8,GRE CARUARU,26050048,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO BENTO AMÉ...
9,GRE CARUARU,26052776,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO CORSINA B...


# PASSO 6 — Aplicar função de extração dos dados

In [ ]:
# ============================================================
# PASSO 6 — FUNÇÕES AUXILIARES E EXTRAÇÃO DOS DADOS VIA REGEX
# ============================================================

import re


def limpar_texto_regex(texto):
    texto = texto or ""
    texto = texto.replace("\r", "\n")
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n+", "\n", texto)
    return texto.strip()


def primeiro_match(texto, padroes):
    for padrao in padroes:
        m = re.search(padrao, texto, re.IGNORECASE | re.DOTALL)
        if m:
            return re.sub(r"\s+", " ", m.group(1)).strip()
    return ""


def normalizar_numero(valor):
    if not valor:
        return ""

    valor = str(valor).strip()
    valor = valor.replace(",", ".")
    valor = re.sub(r"[^\d.]", "", valor)

    return valor if valor else ""


def limpar_nome_escola_regex(nome):
    if not nome:
        return ""

    nome = str(nome)

    # Remove qualquer ruído antes de um código INEP seguido de hífen.
    # Exemplo: "7 | 26117088 - ESCOLA..." -> "ESCOLA..."
    nome = re.sub(
        r"^.*?\b[0-9]{8}[A-Z]?\s*[-–]\s*",
        "",
        nome,
        flags=re.IGNORECASE
    )

    # Remove código INEP no início, se ainda restar.
    nome = re.sub(
        r"^\s*[0-9]{8}[A-Z]?\s*[-–]?\s*",
        "",
        nome,
        flags=re.IGNORECASE
    )

    # Remove símbolos ou ruídos antes do início real do nome.
    nome = re.sub(
        r"^[^A-ZÁÀÃÂÉÊÍÓÔÕÚÇ]+",
        "",
        nome,
        flags=re.IGNORECASE
    )

    # Remove endereço grudado ao nome.
    nome = re.sub(
        r"\s+(AV|AV\.|AVENIDA|R|R\.|RR|RUA|EST|ESTRADA|ENG|PR|PRAÇA|PRACA|SITIO|SÍTIO|ROD|RODOVIA|TRAVESSA)\b.*$",
        "",
        nome,
        flags=re.IGNORECASE
    )

    # Remove campos seguintes caso venham grudados.
    nome = re.sub(
        r"\b(GRE|nRF|GFE|GRF)\b.*$",
        "",
        nome,
        flags=re.IGNORECASE
    )

    nome = re.sub(
        r"\b(Fornecimento\s+do\s+produto|Medida\s+de\s+Fornecimento|Total)\b.*$",
        "",
        nome,
        flags=re.IGNORECASE
    )

    nome = re.sub(r"\s+[=|:;.,\-–—]+$", "", nome)
    nome = re.sub(r"\s+\b(E|R|AV|EST|PR)\b$", "", nome, flags=re.IGNORECASE)

    nome = re.sub(r"\s+", " ", nome)
    nome = nome.strip(" .:-|,'\"“”")

    return nome


def limpar_gre_regex(nome_gre):
    if not nome_gre:
        return ""

    nome_gre = str(nome_gre).upper()

    # Limpeza estrutural apenas.
    # Não corrige palavras específicas para não viciar o experimento.
    nome_gre = re.sub(r"[^A-ZÁÀÃÂÉÊÍÓÔÕÚÇ\s]", " ", nome_gre)
    nome_gre = re.sub(r"\s+", " ", nome_gre).strip()

    if nome_gre and not nome_gre.startswith("GRE"):
        nome_gre = "GRE " + nome_gre

    return nome_gre


def limpar_produto_regex(produto):
    if not produto:
        return ""

    produto = str(produto).upper()

    # Remove campos seguintes caso venham grudados ao produto por erro de OCR.
    produto = re.sub(
        r"\b(MEDIDA\s+DE\s+FORNECIMENTO|TOTAL|DEVOLUÇÃO|DEVOLUCAO|JUSTIFICATIVA|OBSERVAÇÕES|OBSERVACOES)\b.*$",
        "",
        produto,
        flags=re.IGNORECASE
    )

    # Limpeza estrutural apenas.
    # Não corrige nomes específicos de produto para não viciar o experimento.
    produto = re.sub(r"[^A-ZÁÀÃÂÉÊÍÓÔÕÚÇ0-9\s\-/]", " ", produto)
    produto = re.sub(r"\s+", " ", produto)
    produto = produto.strip(" .:-|,'\"“”")

    return produto


def extrair_codigo_inep_regex(texto, nome_escola_raw=""):
    codigo = primeiro_match(
        texto,
        REGEX_CEASA["codigo_inep"]
    )

    if not codigo:
        codigo = primeiro_match(
            texto,
            [
                r"ESCOLA\s*:\s*.*?\b([0-9]{8}[A-Z]?)\s*[-–]",
                r"\b([0-9]{8}[A-Z]?)\s*[-–]\s*ESCOLA"
            ]
        )

    if not codigo and nome_escola_raw:
        codigo = primeiro_match(
            nome_escola_raw,
            [
                r"\b([0-9]{8}[A-Z]?)\s*[-–]"
            ]
        )

    if not codigo:
        return ""

    return codigo.strip().upper()


def extrair_campos_ceasa(texto, pagina=None):
    texto = limpar_texto_regex(texto)

    nome_escola_raw = primeiro_match(
        texto,
        REGEX_CEASA["nome_escola"]
    )

    codigo_inep = extrair_codigo_inep_regex(
        texto=texto,
        nome_escola_raw=nome_escola_raw
    )

    nome_gre = primeiro_match(
        texto,
        REGEX_CEASA["nome_gre"]
    )

    produto = primeiro_match(
        texto,
        REGEX_CEASA["produto"]
    )

    total_kg = primeiro_match(
        texto,
        REGEX_CEASA["total_kg"]
    )

    data_inicio_obs = primeiro_match(
        texto,
        REGEX_CEASA["data_inicio_obs"]
    )

    data_fim_obs = primeiro_match(
        texto,
        REGEX_CEASA["data_fim_obs"]
    )

    resultado = {
        "pagina": pagina,
        "nome_escola": limpar_nome_escola_regex(nome_escola_raw),
        "codigo_inep": codigo_inep,
        "nome_gre": limpar_gre_regex(nome_gre),
        "produto": limpar_produto_regex(produto),
        "total_kg": normalizar_numero(total_kg),
        "data_inicio_obs": data_inicio_obs,
        "data_fim_obs": data_fim_obs
    }

    return {campo: resultado.get(campo, "") for campo in CAMPOS_SAIDA}


def extrair_campos_padrao(texto, pagina=None, page_img=None):
    return extrair_campos_ceasa(
        texto=texto,
        pagina=pagina
    )


print("PASSO 6 corrigido: funções auxiliares carregadas sem substituições específicas.")


PASSO 6 corrigido: funções auxiliares carregadas sem substituições específicas.


# PASSO 7 — Salvar resultado bruto da ferramenta

In [ ]:
inicio = time.time()

paginas_extraidas = extrair_textos_pymupdf_ocr(pdf_path)

resultado_extraido = []

for item in paginas_extraidas:
    resultado = extrair_campos_padrao(
        texto=item["texto"],
        pagina=item["pagina"],
        page_img=item["imagem"]
    )

    resultado_extraido.append(resultado)

tempo_execucao = time.time() - inicio

nome_arquivo_saida = f"resultado_{NOME_FERRAMENTA}.json"

with open(nome_arquivo_saida, "w", encoding="utf-8") as f:
    json.dump(resultado_extraido, f, ensure_ascii=False, indent=4)

print("Extração estruturada finalizada.")
print("Ferramenta:", NOME_FERRAMENTA)
print("Total de páginas processadas:", len(resultado_extraido))
print("Tempo de execução:", round(tempo_execucao, 2), "segundos")
print("Arquivo salvo:", nome_arquivo_saida)

print("\nPrévia dos 3 primeiros resultados:\n")
for item in resultado_extraido[:3]:
    print(json.dumps(item, ensure_ascii=False, indent=4))
    print("-" * 50)

files.download(nome_arquivo_saida)

Página 1 processada
Página 2 processada
Página 3 processada
Página 4 processada
Página 5 processada
Página 6 processada
Página 7 processada
Página 8 processada
Página 9 processada
Página 10 processada
Página 11 processada
Página 12 processada
Página 13 processada
Página 14 processada
Página 15 processada
Página 16 processada
Página 17 processada
Página 18 processada
Página 19 processada
Página 20 processada
Página 21 processada
Página 22 processada
Página 23 processada
Página 24 processada
Página 25 processada
Página 26 processada
Página 27 processada
Página 28 processada
Página 29 processada
Página 30 processada
Página 31 processada
Página 32 processada
Página 33 processada
Página 34 processada
Página 35 processada
Página 36 processada
Página 37 processada
Página 38 processada
Página 39 processada
Página 40 processada
Página 41 processada
Página 42 processada
Página 43 processada
Página 44 processada
Página 45 processada
Página 46 processada
Página 47 processada
Página 48 processada
P

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# PASSO 8 — Aplicar pós-processamento e salvar novo resultado

In [ ]:
# ============================================================
# PASSO 8 — PÓS-PROCESSAR ARQUIVO BRUTO E GERAR NOVO JSON
# ============================================================

import os
import json
from google.colab import files

# ------------------------------------------------------------
# 1. VERIFICAÇÕES INICIAIS
# ------------------------------------------------------------

if "base_escolas" not in globals():
    raise NameError(
        "A variável 'base_escolas' não foi encontrada. "
        "Execute o Passo 2.1 e o Passo 5 antes deste passo."
    )

if "pos_processar_registro" not in globals():
    raise NameError(
        "A função 'pos_processar_registro' não foi encontrada. "
        "Execute o Passo 5 antes deste passo."
    )


# ------------------------------------------------------------
# 2. DEFINE O ARQUIVO BRUTO ESPERADO
# ------------------------------------------------------------

nome_arquivo_bruto = f"resultado_{NOME_FERRAMENTA}.json"

print("Arquivo bruto esperado:", nome_arquivo_bruto)

# ------------------------------------------------------------
# 3. SE O ARQUIVO BRUTO NÃO ESTIVER NO AMBIENTE, FAZ UPLOAD
# ------------------------------------------------------------

if not os.path.exists(nome_arquivo_bruto):
    print("\nArquivo bruto não encontrado no ambiente.")
    print("Envie agora o arquivo bruto gerado no Passo 6.")
    print("Atenção: envie o arquivo resultado_pymupdf_tesseract.json, não o pós-processado.")

    uploaded_json = files.upload()

    arquivo_json_enviado = None

    for nome_arquivo in uploaded_json.keys():
        if nome_arquivo.lower().endswith(".json"):
            arquivo_json_enviado = nome_arquivo
            break

    if arquivo_json_enviado is None:
        raise ValueError("Nenhum arquivo JSON foi enviado.")

    nome_arquivo_bruto = arquivo_json_enviado

print("\nArquivo bruto usado:", nome_arquivo_bruto)

# ------------------------------------------------------------
# 4. BLOQUEIA USO ACIDENTAL DE ARQUIVO JÁ PÓS-PROCESSADO
# ------------------------------------------------------------

if "pos_processado" in nome_arquivo_bruto.lower():
    raise ValueError(
        "Você enviou um arquivo que parece já estar pós-processado. "
        "Envie o arquivo bruto gerado no Passo 6."
    )

# ------------------------------------------------------------
# 5. CARREGA O RESULTADO BRUTO
# ------------------------------------------------------------

with open(nome_arquivo_bruto, "r", encoding="utf-8") as f:
    resultado_bruto = json.load(f)

if not isinstance(resultado_bruto, list):
    raise ValueError("O arquivo bruto precisa conter uma lista de registros.")

if len(resultado_bruto) == 0:
    raise ValueError("O arquivo bruto está vazio.")

print("Total de registros no arquivo bruto:", len(resultado_bruto))

# ------------------------------------------------------------
# 6. VERIFICA SE O CONTEÚDO JÁ TEM _pos_processamento
# ------------------------------------------------------------

registros_com_pos = 0

for registro in resultado_bruto:
    if isinstance(registro, dict) and "_pos_processamento" in registro:
        registros_com_pos += 1

if registros_com_pos > 0:
    raise ValueError(
        f"O arquivo carregado já possui '_pos_processamento' em {registros_com_pos} registro(s). "
        "Isso indica que ele já foi pós-processado. "
        "Use o JSON bruto gerado diretamente no Passo 6."
    )

# ------------------------------------------------------------
# 7. APLICA PÓS-PROCESSAMENTO SEM ALTERAR resultado_extraido
# ------------------------------------------------------------

resultado_pos_processado = []

for registro in resultado_bruto:
    registro_corrigido = pos_processar_registro(
        registro=registro,
        base_escolas=base_escolas
    )

    resultado_pos_processado.append(registro_corrigido)

# ------------------------------------------------------------
# 8. SALVA ARQUIVO PÓS-PROCESSADO
# ------------------------------------------------------------

nome_arquivo_pos = f"resultado_pos_processado_{NOME_FERRAMENTA}.json"

with open(nome_arquivo_pos, "w", encoding="utf-8") as f:
    json.dump(resultado_pos_processado, f, ensure_ascii=False, indent=4)

# ------------------------------------------------------------
# 9. RESUMO DO PÓS-PROCESSAMENTO
# ------------------------------------------------------------

total_registros = len(resultado_pos_processado)
total_base_encontrada = 0
total_gre_corrigida = 0
total_produto_corrigido = 0
total_escola_corrigida = 0
total_codigo_corrigido = 0
total_gre_base_corrigida = 0

for registro in resultado_pos_processado:
    auditoria = registro.get("_pos_processamento", {})

    if auditoria.get("base_encontrada", False):
        total_base_encontrada += 1

    if auditoria.get("gre_corrigida", False):
        total_gre_corrigida += 1

    if auditoria.get("produto_corrigido", False):
        total_produto_corrigido += 1

    if auditoria.get("escola_corrigida_por_base", False):
        total_escola_corrigida += 1

    if auditoria.get("codigo_inep_corrigido_por_base", False):
        total_codigo_corrigido += 1

    if auditoria.get("gre_corrigida_por_base", False):
        total_gre_base_corrigida += 1

print("\nPós-processamento finalizado.")
print("Ferramenta:", NOME_FERRAMENTA)
print("Total de registros processados:", total_registros)
print("Base encontrada:", total_base_encontrada)
print("GRE corrigida por normalização:", total_gre_corrigida)
print("Produto corrigido:", total_produto_corrigido)
print("Escola corrigida pela base:", total_escola_corrigida)
print("Código INEP corrigido pela base:", total_codigo_corrigido)
print("GRE corrigida pela base:", total_gre_base_corrigida)
print("Arquivo pós-processado salvo:", nome_arquivo_pos)

# ------------------------------------------------------------
# 10. PRÉVIA DOS 3 PRIMEIROS RESULTADOS PÓS-PROCESSADOS
# ------------------------------------------------------------

print("\nPrévia dos 3 primeiros resultados pós-processados:\n")

for item in resultado_pos_processado[:3]:
    print(json.dumps(item, ensure_ascii=False, indent=4))
    print("-" * 50)

# ------------------------------------------------------------
# 11. DOWNLOAD DO ARQUIVO PÓS-PROCESSADO
# ------------------------------------------------------------

files.download(nome_arquivo_pos)

Arquivo bruto esperado: resultado_pymupdf_tesseract.json

Arquivo bruto usado: resultado_pymupdf_tesseract.json
Total de registros no arquivo bruto: 164

Pós-processamento finalizado.
Ferramenta: pymupdf_tesseract
Total de registros processados: 164
Base encontrada: 155
GRE corrigida por normalização: 13
Produto corrigido: 8
Escola corrigida pela base: 88
Código INEP corrigido pela base: 5
GRE corrigida pela base: 0
Arquivo pós-processado salvo: resultado_pos_processado_pymupdf_tesseract.json

Prévia dos 3 primeiros resultados pós-processados:

{
    "pagina": 1,
    "nome_escola": "ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO COSTA DE AZEVEDO",
    "codigo_inep": "26098270",
    "nome_gre": "GRE PALMARES",
    "produto": "FRANGO COXA E SOBRECOXA",
    "total_kg": "281.0",
    "data_inicio_obs": "07/04",
    "data_fim_obs": "29/04",
    "_pos_processamento": {
        "base_encontrada": true,
        "gre_corrigida": false,
        "produto_corrigido": false,
        "escola_corrigida_por_base

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# PASSO 9 — Gerar relatório de correções (Excel + PDF)

In [ ]:
# ============================================================
# PASSO 6.1.1 — RELATÓRIO DE CORREÇÕES PARA O USUÁRIO FINAL
# ============================================================
# Gera um arquivo XLSX com duas abas:
#   Aba 1 — "Correções por Campo": detalha cada campo alterado,
#            com valor original, valor corrigido e método usado.
#   Aba 2 — "Resumo por Tipo": conta quantas correções
#            ocorreram por tipo (GRE, produto, escola, INEP).
# ============================================================

!pip install openpyxl --quiet

import pandas as pd
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import files

# ------------------------------------------------------------
# RÓTULOS LEGÍVEIS PARA OS CAMPOS (para o usuário final)
# ------------------------------------------------------------

ROTULOS_CAMPO = {
    "nome_escola":   "Nome da Escola",
    "nome_gre":      "GRE",
    "codigo_inep":   "Código INEP",
    "produto":       "Produto",
    "total_kg":      "Total (KG)",
    "data_inicio_obs": "Data Início",
    "data_fim_obs":  "Data Fim",
    "nome_gre_antes_base": "GRE (antes da base)"
}

ROTULOS_METODO = {
    "codigo_inep+gre":      "INEP + GRE (correspondência exata)",
    "codigo_inep":          "INEP (correspondência exata)",
    "similaridade_nome+gre":"Similaridade de nome dentro da GRE",
    "similaridade_nome":    "Similaridade de nome (base completa)"
}

# ------------------------------------------------------------
# 1. COLETA OS DETALHES DE CADA CORREÇÃO
# ------------------------------------------------------------

detalhes_correcoes = []
paginas_sem_correcao = []

for registro in resultado_pos_processado:
    pagina        = registro.get("pagina", "")
    auditoria     = registro.get("_pos_processamento", {})
    valores_orig  = auditoria.get("valores_originais", {})
    metodo        = auditoria.get("metodo_escola", "")
    similaridade  = auditoria.get("similaridade_escola", "")
    base_encontrada = auditoria.get("base_encontrada", False)

    if not valores_orig:
        paginas_sem_correcao.append(pagina)
        continue

    for campo, valor_original in valores_orig.items():
        # valor corrigido: busca no registro principal (ignora campos internos de auditoria)
        if campo == "nome_gre_antes_base":
            valor_corrigido = registro.get("nome_gre", "")
        else:
            valor_corrigido = registro.get(campo, "")

        # determina a origem da correção
        if campo == "produto":
            origem = "Normalização de produto"
        elif campo == "nome_gre" and not auditoria.get("gre_corrigida_por_base", False):
            origem = "Normalização de GRE"
        elif campo in ("nome_escola", "codigo_inep", "nome_gre", "nome_gre_antes_base"):
            origem = ROTULOS_METODO.get(metodo, metodo) if metodo else "Base de escolas"
        else:
            origem = "Normalização"

        detalhes_correcoes.append({
            "Página":             pagina,
            "Campo Corrigido":    ROTULOS_CAMPO.get(campo, campo),
            "Valor Original":     str(valor_original).strip(),
            "Valor Corrigido":    str(valor_corrigido).strip(),
            "Origem da Correção": origem,
            "Similaridade":       round(float(similaridade), 4) if similaridade != "" and campo in ("nome_escola", "nome_gre_antes_base") else "",
            "Base Encontrada":    "Sim" if base_encontrada else "Não"
        })

df_correcoes = pd.DataFrame(detalhes_correcoes)

# ------------------------------------------------------------
# 2. RESUMO AGREGADO POR TIPO DE CORREÇÃO
# ------------------------------------------------------------

TIPOS_CORRECAO = {
    "GRE corrigida por normalização":    "gre_corrigida",
    "Produto corrigido":                 "produto_corrigido",
    "Escola corrigida pela base":        "escola_corrigida_por_base",
    "Código INEP corrigido pela base":   "codigo_inep_corrigido_por_base",
    "GRE corrigida pela base":           "gre_corrigida_por_base",
}

linhas_resumo_tipo = []

for rotulo, chave in TIPOS_CORRECAO.items():
    quantidade = sum(
        1 for r in resultado_pos_processado
        if r.get("_pos_processamento", {}).get(chave, False)
    )
    total = len(resultado_pos_processado)
    linhas_resumo_tipo.append({
        "Tipo de Correção":   rotulo,
        "Qtd. Páginas":       quantidade,
        "Total de Páginas":   total,
        "% das Páginas":      f"{round(quantidade / total * 100, 1)}%" if total > 0 else "0%"
    })

df_resumo_tipo = pd.DataFrame(linhas_resumo_tipo)

# ------------------------------------------------------------
# 3. SALVA XLSX COM DUAS ABAS FORMATADAS
# ------------------------------------------------------------

nome_relatorio = f"relatorio_correcoes_{NOME_FERRAMENTA}.xlsx"

with pd.ExcelWriter(nome_relatorio, engine="openpyxl") as writer:

    # — Aba 1: Correções por Campo —
    if not df_correcoes.empty:
        df_correcoes.to_excel(writer, sheet_name="Correções por Campo", index=False)
    else:
        pd.DataFrame([{"Informação": "Nenhuma correção registrada."}]).to_excel(
            writer, sheet_name="Correções por Campo", index=False
        )

    # — Aba 2: Resumo por Tipo —
    df_resumo_tipo.to_excel(writer, sheet_name="Resumo por Tipo", index=False)

    wb = writer.book

    # Estilos compartilhados
    cor_cabecalho  = PatternFill("solid", fgColor="1F4E79")
    cor_alterado   = PatternFill("solid", fgColor="FFF2CC")
    fonte_cabecalho = Font(bold=True, color="FFFFFF", size=11)
    fonte_normal    = Font(size=10)
    alinhamento_c   = Alignment(horizontal="center", vertical="center", wrap_text=True)
    alinhamento_e   = Alignment(horizontal="left",   vertical="center", wrap_text=True)
    borda = Border(
        left=Side(style="thin"), right=Side(style="thin"),
        top=Side(style="thin"),  bottom=Side(style="thin")
    )

    for nome_aba in ["Correções por Campo", "Resumo por Tipo"]:
        ws = wb[nome_aba]

        # Formata cabeçalho
        for cell in ws[1]:
            cell.fill      = cor_cabecalho
            cell.font      = fonte_cabecalho
            cell.alignment = alinhamento_c
            cell.border    = borda

        # Formata dados
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.font      = fonte_normal
                cell.alignment = alinhamento_e
                cell.border    = borda

            # Destaca linhas com diferença entre original e corrigido (aba 1)
            if nome_aba == "Correções por Campo" and len(row) >= 4:
                original  = str(row[2].value or "").strip()
                corrigido = str(row[3].value or "").strip()
                if original.upper() != corrigido.upper():
                    for cell in row:
                        cell.fill = cor_alterado

        # Ajusta largura das colunas automaticamente
        for col in ws.columns:
            max_len = max(
                (len(str(cell.value or "")) for cell in col),
                default=10
            )
            ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 60)

        # Congela a primeira linha
        ws.freeze_panes = "A2"

# ------------------------------------------------------------
# 4. EXIBE RESUMO NO CONSOLE E FAZ DOWNLOAD
# ------------------------------------------------------------

print("\nRelatório de Correções — Pós-Processamento")
print("=" * 55)
print(f"  Ferramenta       : {NOME_FERRAMENTA}")
print(f"  Total de páginas : {len(resultado_pos_processado)}")
print(f"  Páginas corrigidas: {len(resultado_pos_processado) - len(paginas_sem_correcao)}")
print(f"  Páginas sem correção: {len(paginas_sem_correcao)}")
print(f"  Total de campos alterados: {len(df_correcoes)}")
print()

if not df_correcoes.empty:
    print("Prévia das primeiras correções registradas:")
    display(df_correcoes.head(10))
else:
    print("Nenhuma correção foi registrada.")

print(f"\nResumo por tipo de correção:")
display(df_resumo_tipo)

print(f"\nArquivo salvo: {nome_relatorio}")
files.download(nome_relatorio)


Relatório de Correções — Pós-Processamento
  Ferramenta       : pymupdf_tesseract
  Total de páginas : 164
  Páginas corrigidas: 99
  Páginas sem correção: 65
  Total de campos alterados: 114

Prévia das primeiras correções registradas:


,Página,Campo Corrigido,Valor Original,Valor Corrigido,Origem da Correção,Similaridade,Base Encontrada
0,1,Nome da Escola,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO COSTA AZE...,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO COSTA DE ...,INEP + GRE (correspondência exata),1.0,Sim
1,2,Nome da Escola,ESCOLA DE REFERENCIA EM ENSINO MEDIO DOUTOR PE...,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO DR. PEDRO...,INEP + GRE (correspondência exata),1.0,Sim
2,4,Nome da Escola,ESCOLA DE REFERENCIA EM ENSINO MEDIO DOUTOR FE...,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO DOUTOR FE...,INEP + GRE (correspondência exata),1.0,Sim
3,8,GRE,GRE JA S N VILA JOSE MARIANO RIBEIRÃO GRE GRE ...,GRE PALMARES,Normalização de GRE,,Sim
4,11,Nome da Escola,ESCOLA DE REFER?NCIA EM ENSINO Mº?DIO JARBAS P...,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO GINÁSIO P...,INEP + GRE (correspondência exata),1.0,Sim
5,12,Nome da Escola,ESCOLA DE REFERÊNCIA EM ENSINO MEDIO JOSE VILE...,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO JOSÉ VILELA,INEP + GRE (correspondência exata),1.0,Sim
6,13,Nome da Escola,ESCOLA DE REFERENCIA EM ENSINO MEDIO DOM VITAL ES,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO DOM VITAL,INEP + GRE (correspondência exata),1.0,Sim
7,15,Nome da Escola,ESCOLA DE REFERENCIA EM ENSINO FUNDAMENTAL E E...,ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO ARQUIPÉLA...,INEP + GRE (correspondência exata),1.0,Sim
8,16,Nome da Escola,CENTRO DE ATENDIMENTO EDUCACIONAL ESPECIALIZAD...,CENTRO DE ATENDIMENTO EDUCACIONAL ESPECIALIZAD...,INEP + GRE (correspondência exata),1.0,Sim
9,18,Produto,FRANGO COXA E SOBRECOXA O,FRANGO COXA E SOBRECOXA,Normalização de produto,,Sim



Resumo por tipo de correção:


,Tipo de Correção,Qtd. Páginas,Total de Páginas,% das Páginas
0,GRE corrigida por normalização,13,164,7.9%
1,Produto corrigido,8,164,4.9%
2,Escola corrigida pela base,88,164,53.7%
3,Código INEP corrigido pela base,5,164,3.0%
4,GRE corrigida pela base,0,164,0.0%



Arquivo salvo: relatorio_correcoes_pymupdf_tesseract.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# PASSO 6.1.2 — RELATÓRIO VISUAL EM PDF
# ============================================================
# Gera um PDF com 4 seções:
#   Capa     — cards com métricas gerais do processamento
#   Seção 1  — motivos de cada tipo de correção
#   Seção 2  — resumo quantitativo com barras e tabela
#   Seção 3  — detalhamento campo a campo por página
#   Seção 4  — páginas que não precisaram de correção
#
# Depende de: resultado_pos_processado e NOME_FERRAMENTA
# (já gerados nos passos anteriores — sem necessidade de upload)
# ============================================================

!pip install reportlab --quiet

import re
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm, cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable, PageBreak, KeepTogether
)
from reportlab.platypus.flowables import Flowable

# ------------------------------------------------------------
# PALETA DE CORES
# ------------------------------------------------------------
_AZUL_ESCURO   = colors.HexColor("#0D2137")
_AZUL_MEDIO    = colors.HexColor("#1A4A7A")
_AZUL_CLARO    = colors.HexColor("#2E86C1")
_VERDE         = colors.HexColor("#1E8449")
_VERDE_CLARO   = colors.HexColor("#D5F5E3")
_AMARELO       = colors.HexColor("#D4AC0D")
_AMARELO_CLARO = colors.HexColor("#FEF9E7")
_LARANJA       = colors.HexColor("#CA6F1E")
_LARANJA_CLARO = colors.HexColor("#FDEBD0")
_CINZA_ESCURO  = colors.HexColor("#2C3E50")
_CINZA_MEDIO   = colors.HexColor("#717D7E")
_CINZA_CLARO   = colors.HexColor("#F2F3F4")
_CINZA_LINHA   = colors.HexColor("#E5E8E8")
_BRANCO        = colors.white
_W, _H = A4

# ------------------------------------------------------------
# MAPEAMENTOS
# ------------------------------------------------------------
_ROTULOS_CAMPO = {
    "nome_escola":       "Nome da Escola",
    "nome_gre":          "GRE",
    "codigo_inep":       "Código INEP",
    "produto":           "Produto",
    "total_kg":          "Total (KG)",
    "data_inicio_obs":   "Data Início",
    "data_fim_obs":      "Data Fim",
    "nome_gre_antes_base": "GRE (antes da base)"
}
_ROTULOS_METODO = {
    "codigo_inep+gre":       "INEP + GRE (correspondência exata)",
    "codigo_inep":           "INEP (correspondência exata)",
    "similaridade_nome+gre": "Similaridade de nome dentro da GRE",
    "similaridade_nome":     "Similaridade de nome (base completa)"
}
_COR_CAMPO = {
    "Nome da Escola":      _AZUL_CLARO,
    "GRE":                 _VERDE,
    "GRE (antes da base)": _VERDE,
    "Produto":             _AMARELO,
    "Código INEP":         _LARANJA,
}

# ------------------------------------------------------------
# FLOWABLES CUSTOMIZADOS
# ------------------------------------------------------------
class _CardMetrica(Flowable):
    def __init__(self, label, valor, cor, largura=40*mm):
        super().__init__()
        self.label  = label
        self.valor  = str(valor)
        self.cor    = cor
        self.width  = largura
        self.height = 22*mm

    def draw(self):
        c = self.canv
        larg = self.width - 2
        c.setFillColor(colors.HexColor("#DADDE1"))
        c.roundRect(1, -1, larg, self.height, 4, fill=1, stroke=0)
        c.setFillColor(_BRANCO)
        c.roundRect(0, 0, larg, self.height, 4, fill=1, stroke=0)
        c.setFillColor(self.cor)
        c.roundRect(0, self.height - 5*mm, larg, 5*mm, 4, fill=1, stroke=0)
        c.rect(0, self.height - 5*mm, larg, 2*mm, fill=1, stroke=0)
        c.setFillColor(self.cor)
        c.setFont("Helvetica-Bold", 22)
        c.drawCentredString(larg / 2, 7*mm, self.valor)
        c.setFillColor(_CINZA_MEDIO)
        c.setFont("Helvetica", 7)
        c.drawCentredString(larg / 2, 3*mm, self.label)


class _BarraProgresso(Flowable):
    def __init__(self, label, quantidade, total, cor):
        super().__init__()
        self.label      = label
        self.quantidade = quantidade
        self.total      = total
        self.cor        = cor
        self.width      = _W - 4*cm
        self.height     = 9*mm

    def draw(self):
        c   = self.canv
        pct = self.quantidade / self.total if self.total > 0 else 0
        barra_tot   = self.width - 70*mm
        barra_preen = barra_tot * pct
        c.setFillColor(_CINZA_ESCURO)
        c.setFont("Helvetica", 8.5)
        c.drawString(0, 3*mm, self.label)
        c.setFillColor(_CINZA_LINHA)
        c.roundRect(55*mm, 2*mm, barra_tot, 5*mm, 2, fill=1, stroke=0)
        if barra_preen > 0:
            c.setFillColor(self.cor)
            c.roundRect(55*mm, 2*mm, barra_preen, 5*mm, 2, fill=1, stroke=0)
        c.setFillColor(_CINZA_ESCURO)
        c.setFont("Helvetica-Bold", 8.5)
        c.drawRightString(self.width, 3*mm, f"{self.quantidade}  ({pct*100:.1f}%)")


# ------------------------------------------------------------
# CABEÇALHO E RODAPÉ
# ------------------------------------------------------------
class _Numerador:
    def __init__(self, ferramenta, data_geracao):
        self.ferramenta   = ferramenta
        self.data_geracao = data_geracao

    def __call__(self, canv, doc):
        canv.saveState()
        canv.setFillColor(_AZUL_ESCURO)
        canv.rect(0, _H - 18*mm, _W, 18*mm, fill=1, stroke=0)
        canv.setFillColor(_BRANCO)
        canv.setFont("Helvetica-Bold", 11)
        canv.drawString(2*cm, _H - 11*mm, "RELATÓRIO DE PÓS-PROCESSAMENTO OCR")
        canv.setFont("Helvetica", 8)
        canv.drawRightString(_W - 2*cm, _H - 11*mm, f"Ferramenta: {self.ferramenta}")
        canv.setStrokeColor(_AZUL_CLARO)
        canv.setLineWidth(2)
        canv.line(0, _H - 18*mm, _W, _H - 18*mm)
        canv.setFillColor(_CINZA_CLARO)
        canv.rect(0, 0, _W, 12*mm, fill=1, stroke=0)
        canv.setStrokeColor(_CINZA_LINHA)
        canv.setLineWidth(0.5)
        canv.line(0, 12*mm, _W, 12*mm)
        canv.setFillColor(_CINZA_MEDIO)
        canv.setFont("Helvetica", 7.5)
        canv.drawString(2*cm, 4*mm, f"Gerado em {self.data_geracao}  ·  Projeto OCR CEASA")
        canv.drawRightString(_W - 2*cm, 4*mm, f"Página {doc.page}")
        canv.restoreState()


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def _st(nome, **kw):
    d = dict(fontName="Helvetica", fontSize=9, textColor=_CINZA_ESCURO, spaceAfter=4, leading=13)
    d.update(kw)
    return ParagraphStyle(nome, **d)

def _p(texto, estilo):
    return Paragraph(texto, estilo)

def _hr():
    return HRFlowable(width="100%", thickness=1, color=_AZUL_CLARO, spaceAfter=5)

def _sp(h=3):
    return Spacer(1, h*mm)

def _card_nota(texto, cor_fundo, cor_barra):
    t = Table([[_p(texto, _st("cn", fontName="Helvetica", fontSize=8.5,
                              textColor=_CINZA_ESCURO, leading=13, spaceAfter=0))]],
              colWidths=[_W - 4*cm])
    t.setStyle(TableStyle([
        ("BACKGROUND",   (0,0),(-1,-1), cor_fundo),
        ("TOPPADDING",   (0,0),(-1,-1), 6),
        ("BOTTOMPADDING",(0,0),(-1,-1), 6),
        ("LEFTPADDING",  (0,0),(-1,-1), 8),
        ("RIGHTPADDING", (0,0),(-1,-1), 8),
        ("LINEBEFORE",   (0,0),(0,-1),  4, cor_barra),
    ]))
    return t

# ------------------------------------------------------------
# COLETA DE DADOS A PARTIR DE resultado_pos_processado
# ------------------------------------------------------------
_data_geracao    = datetime.now().strftime("%d/%m/%Y %H:%M")
_total_paginas   = len(resultado_pos_processado)
_correcoes_pdf   = []
_paginas_sem_cor = []
_cont = {
    "escola_corrigida_por_base":      0,
    "gre_corrigida":                  0,
    "produto_corrigido":              0,
    "codigo_inep_corrigido_por_base": 0,
    "gre_corrigida_por_base":         0,
}
_met_cont = {}

for _reg in resultado_pos_processado:
    _aud    = _reg.get("_pos_processamento", {})
    _vorig  = _aud.get("valores_originais", {})
    _metodo = _aud.get("metodo_escola", "")
    _sim    = _aud.get("similaridade_escola", "")

    for _chave in _cont:
        if _aud.get(_chave, False):
            _cont[_chave] += 1

    if _vorig:
        _rot_met = _ROTULOS_METODO.get(_metodo, _metodo) if _metodo else "Normalização"
        _met_cont[_rot_met] = _met_cont.get(_rot_met, 0) + 1

        for _campo, _val_orig in _vorig.items():
            _val_corr = _reg.get("nome_gre", "") if _campo == "nome_gre_antes_base" else _reg.get(_campo, "")
            if _campo == "produto":
                _origem = "Normalização de produto"
            elif _campo == "nome_gre" and not _aud.get("gre_corrigida_por_base", False):
                _origem = "Normalização de GRE"
            elif _campo in ("nome_escola", "codigo_inep", "nome_gre", "nome_gre_antes_base"):
                _origem = _ROTULOS_METODO.get(_metodo, _metodo) if _metodo else "Base de escolas"
            else:
                _origem = "Normalização"

            _sim_val = ""
            if _sim != "" and _campo in ("nome_escola", "nome_gre_antes_base"):
                try:
                    _sim_val = round(float(_sim), 4)
                except Exception:
                    pass

            _correcoes_pdf.append({
                "pagina":       _reg.get("pagina", ""),
                "campo":        _ROTULOS_CAMPO.get(_campo, _campo),
                "original":     str(_val_orig).strip(),
                "corrigido":    str(_val_corr).strip(),
                "metodo":       _origem,
                "similaridade": _sim_val,
            })
    else:
        _paginas_sem_cor.append(_reg.get("pagina", ""))

_paginas_com_correcao = _total_paginas - len(_paginas_sem_cor)
_total_campos_alt     = len(_correcoes_pdf)
_resumo_tipo = [
    {"tipo": "Nome da Escola corrigido pela base",  "qt": _cont["escola_corrigida_por_base"],      "cor": _AZUL_CLARO},
    {"tipo": "GRE corrigida por normalização",      "qt": _cont["gre_corrigida"],                  "cor": _VERDE},
    {"tipo": "Produto corrigido",                   "qt": _cont["produto_corrigido"],               "cor": _AMARELO},
    {"tipo": "Código INEP corrigido pela base",     "qt": _cont["codigo_inep_corrigido_por_base"],  "cor": _LARANJA},
    {"tipo": "GRE corrigida pela base",             "qt": _cont["gre_corrigida_por_base"],          "cor": _AZUL_MEDIO},
]
_metodos_busca = [
    {"metodo": met, "qt": qt,
     "pct": round(qt / _paginas_com_correcao * 100, 1) if _paginas_com_correcao > 0 else 0}
    for met, qt in _met_cont.items()
]

# ------------------------------------------------------------
# MONTAGEM DO STORY
# ------------------------------------------------------------
_story = []

# — CAPA —
_capa_inner = [
    [_p("RELATÓRIO DE PÓS-PROCESSAMENTO", _st("ci", fontName="Helvetica-Bold", fontSize=8,
         textColor=colors.HexColor("#7FB3D3"), alignment=TA_CENTER, spaceAfter=4))],
    [_p("Correções Aplicadas na<br/>Extração OCR", _st("ct", fontName="Helvetica-Bold",
         fontSize=22, textColor=_BRANCO, alignment=TA_CENTER, leading=28, spaceAfter=4))],
    [_p(f"Ferramenta: {NOME_FERRAMENTA.replace('_',' ').title()}", _st("cs",
         fontName="Helvetica", fontSize=10, textColor=colors.HexColor("#A9CCE3"), alignment=TA_CENTER))],
    [_p(f"Gerado em {_data_geracao}", _st("cd", fontName="Helvetica", fontSize=9,
         textColor=colors.HexColor("#85929E"), alignment=TA_CENTER))],
]
_tc = Table(_capa_inner, colWidths=[_W - 4*cm])
_tc.setStyle(TableStyle([
    ("BACKGROUND",   (0,0),(-1,-1), _AZUL_ESCURO),
    ("TOPPADDING",   (0,0),(-1,-1), 6),
    ("BOTTOMPADDING",(0,0),(-1,-1), 6),
    ("LEFTPADDING",  (0,0),(-1,-1), 12),
    ("RIGHTPADDING", (0,0),(-1,-1), 12),
    ("ROUNDEDCORNERS", [6]),
]))
_story += [_sp(5), _tc, _sp(6)]

_cards = [
    _CardMetrica("Páginas totais",     _total_paginas,        _AZUL_MEDIO, 40*mm),
    _CardMetrica("Páginas corrigidas", _paginas_com_correcao, _AZUL_CLARO, 40*mm),
    _CardMetrica("Sem alteração",      len(_paginas_sem_cor), _VERDE,      40*mm),
    _CardMetrica("Campos alterados",   _total_campos_alt,     _LARANJA,    40*mm),
]
_tc2 = Table([[_cards]], colWidths=[_W - 4*cm])
_tc2.setStyle(TableStyle([("ALIGN",(0,0),(-1,-1),"CENTER"),("VALIGN",(0,0),(-1,-1),"MIDDLE")]))
_story += [_tc2, _sp(3)]
_pct_cor = round(_paginas_com_correcao / _total_paginas * 100, 1) if _total_paginas > 0 else 0
_story += [_p(f"<b>{_pct_cor}%</b> das páginas tiveram ao menos um campo ajustado.",
    _st("pcc", fontName="Helvetica", fontSize=9, textColor=_CINZA_MEDIO, alignment=TA_CENTER)),
    PageBreak()]

# — SEÇÃO 1: Por que o pós-processamento existe? —
_story += [_sp(3),
    _p("1 — Por que o pós-processamento existe?",
       _st("ts1", fontName="Helvetica-Bold", fontSize=13, textColor=_AZUL_ESCURO,
           spaceAfter=4, spaceBefore=12, leading=16)), _hr()]
_explicacoes_pdf = [
    ("🔤  Erros de OCR em caracteres especiais", _AZUL_CLARO, colors.HexColor("#D6EAF8"),
     "O Tesseract falha frequentemente em acentos e cedilhas. Caracteres como ç, ã, é surgem como ? "
     "ou letras incorretas. O pós-processamento corrige comparando com a base oficial de escolas."),
    ("📎  Informações extras capturadas junto ao nome", _VERDE, _VERDE_CLARO,
     "O OCR lê linha a linha e muitas vezes captura endereço ou GRE colados ao nome da escola. "
     "A base permite identificar o nome correto e descartar o ruído."),
    ("🏷️  Normalização de GRE e Produto", _AMARELO, _AMARELO_CLARO,
     "Variações como 'GRE JA S N VILA JOSE MARIANO RIBEIRAO' são normalizadas para 'GRE PALMARES'. "
     "'FRANGO COXA E SOBRECOXA O' é corrigido via dicionário de substituições."),
    ("🔢  Código INEP divergente", _LARANJA, _LARANJA_CLARO,
     "Quando o INEP extraído difere do cadastro oficial e o nome tem similaridade >= 82% "
     "com um registro da base dentro da GRE, o código correto é aplicado."),
]
for _t_e, _c_e, _b_e, _x_e in _explicacoes_pdf:
    _ce = Table([
        [_p(_t_e, _st("et", fontName="Helvetica-Bold", fontSize=9.5, textColor=_c_e, spaceAfter=3, leading=12))],
        [_p(_x_e, _st("eb", fontName="Helvetica", fontSize=8.5, textColor=_CINZA_ESCURO, leading=13))],
    ], colWidths=[_W - 4*cm])
    _ce.setStyle(TableStyle([
        ("BACKGROUND",   (0,0),(-1,-1), _b_e),
        ("TOPPADDING",   (0,0),(-1,-1), 5),
        ("BOTTOMPADDING",(0,0),(-1,-1), 6),
        ("LEFTPADDING",  (0,0),(-1,-1), 8),
        ("RIGHTPADDING", (0,0),(-1,-1), 8),
        ("LINEBEFORE",   (0,0),(0,-1),  3, _c_e),
        ("ROUNDEDCORNERS", [3]),
    ]))
    _story += [_ce, _sp(2)]
_story.append(PageBreak())

# — SEÇÃO 2: Resumo quantitativo —
_story += [_sp(3),
    _p("2 — Resumo quantitativo das correções",
       _st("ts2", fontName="Helvetica-Bold", fontSize=13, textColor=_AZUL_ESCURO,
           spaceAfter=4, spaceBefore=12, leading=16)), _hr(),
    _p("Correções por tipo", _st("st2", fontName="Helvetica-Bold", fontSize=10,
       textColor=_AZUL_MEDIO, spaceAfter=3, leading=13)), _sp(1)]
_tot_tipo = sum(r["qt"] for r in _resumo_tipo) or 1
for _it in _resumo_tipo:
    _story += [_BarraProgresso(_it["tipo"], _it["qt"], _tot_tipo, _it["cor"]), _sp(1)]

_story += [_sp(5),
    _p("Métodos de busca utilizados", _st("st3", fontName="Helvetica-Bold", fontSize=10,
       textColor=_AZUL_MEDIO, spaceAfter=3, leading=13)), _sp(1)]

_conf_map = {
    "INEP + GRE (correspondência exata)":   ("Alta",  _VERDE),
    "INEP (correspondência exata)":          ("Alta",  _VERDE),
    "Similaridade de nome dentro da GRE":   ("Média", _AMARELO),
    "Similaridade de nome (base completa)": ("Baixa", _LARANJA),
    "Normalização":                         ("—",     _CINZA_MEDIO),
    "Normalização de produto":              ("—",     _CINZA_MEDIO),
    "Normalização de GRE":                  ("—",     _CINZA_MEDIO),
    "Base de escolas":                      ("Alta",  _VERDE),
}
_cab_m = [
    _p("Método de Busca", _st("mh0", fontName="Helvetica-Bold", fontSize=8.5, textColor=_BRANCO)),
    _p("Qtd.",            _st("mh1", fontName="Helvetica-Bold", fontSize=8.5, textColor=_BRANCO, alignment=TA_CENTER)),
    _p("%",               _st("mh2", fontName="Helvetica-Bold", fontSize=8.5, textColor=_BRANCO, alignment=TA_CENTER)),
    _p("Confiança",       _st("mh3", fontName="Helvetica-Bold", fontSize=8.5, textColor=_BRANCO, alignment=TA_CENTER)),
]
_lins_m = [_cab_m]
for _im, _m in enumerate(_metodos_busca):
    _ct, _cc = _conf_map.get(_m["metodo"], ("—", _CINZA_MEDIO))
    _lins_m.append([
        _p(_m["metodo"],    _st(f"ma{_im}", fontName="Helvetica", fontSize=8.5, textColor=_CINZA_ESCURO)),
        _p(str(_m["qt"]),   _st(f"mb{_im}", fontName="Helvetica-Bold", fontSize=9, textColor=_AZUL_CLARO, alignment=TA_CENTER)),
        _p(f"{_m['pct']}%", _st(f"mc{_im}", fontName="Helvetica", fontSize=8.5, textColor=_CINZA_ESCURO, alignment=TA_CENTER)),
        _p(f"● {_ct}",      _st(f"md{_im}", fontName="Helvetica-Bold", fontSize=8.5, textColor=_cc, alignment=TA_CENTER)),
    ])
_cw_m = [(_W-4*cm)*p for p in [0.52, 0.12, 0.12, 0.24]]
_tab_m = Table(_lins_m if len(_lins_m) > 1 else [_cab_m, [_p("Sem dados",_st("sd")),"","",""]], colWidths=_cw_m)
_tab_m.setStyle(TableStyle([
    ("BACKGROUND",    (0,0),(-1,0),  _AZUL_ESCURO),
    ("ROWBACKGROUNDS",(0,1),(-1,-1), [_BRANCO, _CINZA_CLARO]),
    ("GRID",          (0,0),(-1,-1), 0.4, _CINZA_LINHA),
    ("TOPPADDING",    (0,0),(-1,-1), 5),
    ("BOTTOMPADDING", (0,0),(-1,-1), 5),
    ("LEFTPADDING",   (0,0),(-1,-1), 6),
    ("RIGHTPADDING",  (0,0),(-1,-1), 6),
    ("VALIGN",        (0,0),(-1,-1), "MIDDLE"),
    ("LINEBELOW",     (0,0),(-1,0),  1, _AZUL_CLARO),
]))
_story += [_tab_m, _sp(3),
    _p("• Alta: encontrado pelo INEP na base.  • Média: similaridade >= 82% dentro da GRE.  "
       "• Baixa: similaridade >= 90% em toda a base.",
       _st("leg", fontName="Helvetica-Oblique", fontSize=7.5, textColor=_CINZA_MEDIO, leading=11)),
    PageBreak()]

# — SEÇÃO 3: Detalhamento por página —
_story += [_sp(3),
    _p("3 — Detalhamento das alterações por página",
       _st("ts3", fontName="Helvetica-Bold", fontSize=13, textColor=_AZUL_ESCURO,
           spaceAfter=4, spaceBefore=12, leading=16)), _hr(),
    _p("Vermelho: extraído pelo OCR.  Verde: corrigido pela base oficial.",
       _st("dl", fontName="Helvetica", fontSize=8.5, textColor=_CINZA_MEDIO, leading=12)), _sp(3)]

_por_pag = {}
for _c in _correcoes_pdf:
    _por_pag.setdefault(_c["pagina"], []).append(_c)

for _pag, _corrs in sorted(_por_pag.items(), key=lambda x: int(str(x[0])) if str(x[0]).isdigit() else 0):
    _hdr = Table([[
        _p(f"Página {_pag}", _st("ph", fontName="Helvetica-Bold", fontSize=10, textColor=_BRANCO)),
        _p(f"{len(_corrs)} campo(s) alterado(s)", _st("pc2", fontName="Helvetica", fontSize=8.5,
           textColor=colors.HexColor("#A9CCE3"), alignment=TA_RIGHT)),
    ]], colWidths=[(_W-4*cm)*0.6, (_W-4*cm)*0.4])
    _hdr.setStyle(TableStyle([
        ("BACKGROUND",   (0,0),(-1,-1), _AZUL_MEDIO),
        ("TOPPADDING",   (0,0),(-1,-1), 4),
        ("BOTTOMPADDING",(0,0),(-1,-1), 4),
        ("LEFTPADDING",  (0,0),(-1,-1), 7),
        ("RIGHTPADDING", (0,0),(-1,-1), 7),
        ("VALIGN",       (0,0),(-1,-1), "MIDDLE"),
        ("ROUNDEDCORNERS", [3]),
    ]))
    _ld = [[
        _p("Campo",          _st("dh0", fontName="Helvetica-Bold", fontSize=7.5, textColor=_CINZA_MEDIO)),
        _p("OCR (original)", _st("dh1", fontName="Helvetica-Bold", fontSize=7.5, textColor=colors.HexColor("#922B21"))),
        _p("Corrigido",      _st("dh2", fontName="Helvetica-Bold", fontSize=7.5, textColor=_VERDE)),
        _p("Método",         _st("dh3", fontName="Helvetica-Bold", fontSize=7.5, textColor=_CINZA_MEDIO)),
    ]]
    for _cr in _corrs:
        _cc2 = _COR_CAMPO.get(_cr["campo"], _AZUL_CLARO)
        _st2 = ""
        if _cr["similaridade"] != "" and _cr["similaridade"] != 1.0:
            try:
                _st2 = f" (sim. {float(_cr['similaridade']):.0%})"
            except Exception:
                pass
        _ld.append([
            _p(f"<b>{_cr['campo']}</b>", _st(f"dca{_pag}", fontName="Helvetica-Bold",
               fontSize=8, textColor=_cc2, leading=11)),
            _p(_cr["original"],  _st(f"dcb{_pag}", fontName="Helvetica", fontSize=7.5,
               textColor=colors.HexColor("#C0392B"), leading=10)),
            _p(_cr["corrigido"], _st(f"dcc{_pag}", fontName="Helvetica-Bold", fontSize=7.5,
               textColor=_VERDE, leading=10)),
            _p(_cr["metodo"] + _st2, _st(f"dcd{_pag}", fontName="Helvetica", fontSize=7,
               textColor=_CINZA_MEDIO, leading=10)),
        ])
    _cw_d = [(_W-4*cm)*p for p in [0.15, 0.33, 0.33, 0.19]]
    _td = Table(_ld, colWidths=_cw_d)
    _td.setStyle(TableStyle([
        ("BACKGROUND",    (0,0),(-1,0),  colors.HexColor("#F8F9FA")),
        ("LINEBELOW",     (0,0),(-1,0),  0.5, _CINZA_LINHA),
        ("ROWBACKGROUNDS",(0,1),(-1,-1), [_BRANCO, colors.HexColor("#FAFAFA")]),
        ("GRID",          (0,1),(-1,-1), 0.3, _CINZA_LINHA),
        ("TOPPADDING",    (0,0),(-1,-1), 4),
        ("BOTTOMPADDING", (0,0),(-1,-1), 4),
        ("LEFTPADDING",   (0,0),(-1,-1), 5),
        ("RIGHTPADDING",  (0,0),(-1,-1), 5),
        ("VALIGN",        (0,0),(-1,-1), "TOP"),
    ]))
    _story.append(KeepTogether([_hdr, _td, _sp(3)]))

_story.append(PageBreak())

# — SEÇÃO 4: Páginas sem alteração —
_story += [_sp(3),
    _p("4 — Páginas sem alterações",
       _st("ts4", fontName="Helvetica-Bold", fontSize=13, textColor=_AZUL_ESCURO,
           spaceAfter=4, spaceBefore=12, leading=16)),
    HRFlowable(width="100%", thickness=1, color=_VERDE, spaceAfter=5),
    _p(f"As {len(_paginas_sem_cor)} páginas abaixo foram extraídas corretamente sem nenhum ajuste.",
       _st("ps", fontName="Helvetica", fontSize=9, textColor=_CINZA_ESCURO, leading=13)), _sp(3)]

_por_linha_b = 10
_lins_b, _lin_b = [], []
for _ib, _pb in enumerate(sorted(_paginas_sem_cor, key=lambda x: int(str(x)) if str(x).isdigit() else 0)):
    _bdg = Table([[_p(str(_pb), _st(f"b{_pb}", fontName="Helvetica-Bold", fontSize=8,
                       textColor=_VERDE, alignment=TA_CENTER))]],
                 colWidths=[12*mm], rowHeights=[7*mm])
    _bdg.setStyle(TableStyle([
        ("BACKGROUND",   (0,0),(-1,-1), _VERDE_CLARO),
        ("ALIGN",        (0,0),(-1,-1), "CENTER"),
        ("VALIGN",       (0,0),(-1,-1), "MIDDLE"),
        ("ROUNDEDCORNERS", [3]),
    ]))
    _lin_b.append(_bdg)
    if len(_lin_b) == _por_linha_b or _ib == len(_paginas_sem_cor)-1:
        while len(_lin_b) < _por_linha_b:
            _lin_b.append("")
        _lins_b.append(_lin_b)
        _lin_b = []

if _lins_b:
    _tb2 = Table(_lins_b, colWidths=[13*mm]*_por_linha_b)
    _tb2.setStyle(TableStyle([
        ("TOPPADDING",   (0,0),(-1,-1), 2),
        ("BOTTOMPADDING",(0,0),(-1,-1), 2),
        ("LEFTPADDING",  (0,0),(-1,-1), 1),
        ("RIGHTPADDING", (0,0),(-1,-1), 1),
    ]))
    _story += [_tb2, _sp(5)]

_pct_ok = round(len(_paginas_sem_cor) / _total_paginas * 100, 1) if _total_paginas > 0 else 0
_story.append(_card_nota(
    f"<b>Taxa de sucesso sem intervenção:</b> {_pct_ok}% das páginas "
    f"({len(_paginas_sem_cor)} de {_total_paginas}) foram extraídas corretamente. "
    "O pós-processamento atuou somente onde necessário.",
    _VERDE_CLARO, _VERDE
))

# ------------------------------------------------------------
# GERAÇÃO DO PDF
# ------------------------------------------------------------
nome_pdf = f"relatorio_pdf_{NOME_FERRAMENTA}.pdf"

_doc_pdf = SimpleDocTemplate(
    nome_pdf,
    pagesize=A4,
    leftMargin=2*cm, rightMargin=2*cm,
    topMargin=2.2*cm, bottomMargin=1.8*cm,
    title="Relatório de Pós-Processamento OCR",
    author="Projeto OCR CEASA",
)
_numerador_pdf = _Numerador(NOME_FERRAMENTA, _data_geracao)
_doc_pdf.build(_story, onFirstPage=_numerador_pdf, onLaterPages=_numerador_pdf)

print(f"\n✅ PDF gerado: {nome_pdf}")
print(f"   Total de páginas processadas : {_total_paginas}")
print(f"   Páginas corrigidas           : {_paginas_com_correcao}")
print(f"   Campos alterados             : {_total_campos_alt}")

files.download(nome_pdf)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.3 MB/s eta 0:00:00

✅ PDF gerado: relatorio_pdf_pymupdf_tesseract.pdf
   Total de páginas processadas : 164
   Páginas corrigidas           : 99
   Campos alterados             : 114


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

# 🔬 BLOCO 2 — BENCHMARK *(somente desenvolvimento)*

> ⚠️ **Este bloco não estará presente em produção.**  
> Ele existe exclusivamente para medir o desempenho da ferramenta durante o estudo de benchmark.  
> Os Passos 10, 11 e 12 dependem de um gabarito (*ground truth*) que não existirá no pipeline final da ARPE.

---

# PASSO 10 — Upload do gabarito

In [ ]:
uploaded_gabarito = files.upload()

gabarito_path = list(uploaded_gabarito.keys())[0]

with open(gabarito_path, "r", encoding="utf-8") as f:
    gabarito = json.load(f)

print("Gabarito carregado:", gabarito_path)
print("Total de registros no gabarito:", len(gabarito))

print("\nPrimeiro registro do gabarito:")
print(gabarito[0])

Saving gabarito_novo_ordenado_corrigido.json to gabarito_novo_ordenado_corrigido.json
Gabarito carregado: gabarito_novo_ordenado_corrigido.json
Total de registros no gabarito: 164

Primeiro registro do gabarito:
{'pagina': 1, 'nome_escola': 'ESCOLA DE REFERÊNCIA EM ENSINO MÉDIO COSTA DE AZEVEDO', 'codigo_inep': '26098270', 'nome_gre': 'GRE PALMARES', 'produto': 'FRANGO COXA E SOBRECOXA', 'total_kg': '281.0', 'data_inicio_obs': '07/04', 'data_fim_obs': '29/04'}


# PASSO 11 — Avaliar resultado pós-processado contra gabarito

In [ ]:
# Usa o resultado pós-processado como entrada da avaliação
# (mede a qualidade do pipeline completo, não apenas do OCR bruto)

import os

JSON_EXTRACAO = f'/content/resultado_pos_processado_{NOME_FERRAMENTA}.json'
JSON_GABARITO = f'/content/{gabarito_path}'  # ground truth

# Coluna de chave de junção (deve existir em ambos os arquivos)
COLUNA_CHAVE = 'pagina'

CAMPOS_SAIDA = [
   "pagina",
    "nome_escola",
    "codigo_inep",
    "nome_gre",
    "produto",
    "total_kg",
    "data_inicio_obs",
    "data_fim_obs"
]

# ============================================================

for f in [JSON_EXTRACAO, JSON_GABARITO]:
    assert os.path.exists(f), (
        f'Arquivo não encontrado: {f}\n'
        'Faça upload no Colab (📁 → Upload).')

print('Arquivos encontrados.')

Arquivos encontrados.


In [ ]:
from IPython.core.display import JSON
import pandas as pd
DF_EXT = pd.read_json(JSON_EXTRACAO, dtype=str)
DF_GAB = pd.read_json(JSON_GABARITO, dtype=str)

DF_EXT[COLUNA_CHAVE] = DF_EXT[COLUNA_CHAVE].astype(int)
DF_GAB[COLUNA_CHAVE] = DF_GAB[COLUNA_CHAVE].astype(int)

print(f'Extração : {len(DF_EXT)} páginas | '
      f'{list(DF_EXT.columns)}')
print(f'Gabarito : {len(DF_GAB)} páginas | '
      f'{list(DF_GAB.columns)}')

faltam_ext = [c for c in CAMPOS_SAIDA if c not in DF_EXT.columns]
faltam_gab = [c for c in CAMPOS_SAIDA if c not in DF_GAB.columns]
if faltam_ext:
    print(f' Campos ausentes na extração : {faltam_ext}')
if faltam_gab:
    print(f' Campos ausentes no gabarito : {faltam_gab}')

# Filtra apenas campos presentes nos dois
CAMPOS_COMUNS = [
    c for c in CAMPOS_SAIDA
    if c in DF_EXT.columns and c in DF_GAB.columns]

print(f'\nCampos a avaliar: {CAMPOS_COMUNS}')

Extração : 164 páginas | ['pagina', 'nome_escola', 'codigo_inep', 'nome_gre', 'produto', 'total_kg', 'data_inicio_obs', 'data_fim_obs', '_pos_processamento']
Gabarito : 164 páginas | ['pagina', 'nome_escola', 'codigo_inep', 'nome_gre', 'produto', 'total_kg', 'data_inicio_obs', 'data_fim_obs']

Campos a avaliar: ['pagina', 'nome_escola', 'codigo_inep', 'nome_gre', 'produto', 'total_kg', 'data_inicio_obs', 'data_fim_obs']


In [ ]:
def normalizar(texto):
    """Normaliza texto para comparação: lowercase, sem espaços extras."""
    if pd.isna(texto) or texto is None:
        return ''
    texto = str(texto).lower().strip()
    texto = re.sub(r'\s+', ' ', texto)
    return texto
def calc_acuracia_exata(gt: str, hyp: str) -> int:
    """1 se extração é idêntica ao GT (após normalização), 0 caso contrário."""
    return int(normalizar(gt) == normalizar(hyp))


print('✅ Funções de métricas prontas.')

✅ Funções de métricas prontas.


In [ ]:
import re
import pandas as pd

JSON_EXTRACAO = f'/content/resultado_pos_processado_{NOME_FERRAMENTA}.json'
JSON_GABARITO = f'/content/{gabarito_path}'

DF_EXT = pd.read_json(JSON_EXTRACAO, dtype=str)
DF_GAB = pd.read_json(JSON_GABARITO, dtype=str)

DF_EXT[COLUNA_CHAVE] = DF_EXT[COLUNA_CHAVE].astype(int)
DF_GAB[COLUNA_CHAVE] = DF_GAB[COLUNA_CHAVE].astype(int)

# Filtra apenas campos presentes nos dois
CAMPOS_COMUNS = [
    c for c in CAMPOS_SAIDA
    if c in DF_EXT.columns and c in DF_GAB.columns]

DF_MERGED = DF_EXT.merge(
    DF_GAB[CAMPOS_COMUNS],
    on=COLUNA_CHAVE,
    how='inner',
    suffixes=('_ocr', '_gt'))

print(f'Páginas com correspondência: {len(DF_MERGED)}')

DATE_FIELDS = ['data_entrega', 'data_inicio', 'data_fim']

# Função de normalização para datas
def normalizar_data(texto):
    """
    Normaliza strings de data para um formato comparável (DD/MM).
    Lida com 'YYYY-MM-DD HH:MM:SS' e 'DD/MM'.
    """
    if pd.isna(texto) or not str(texto).strip():
        return ''
    texto = str(texto).strip()

    try:
        dt = pd.to_datetime(texto)
        return dt.strftime('%d/%m') # Converte para DD/MM
    except ValueError:
        pass # Não é um datetime completo, continua tentando outros formatos

    # Se não for datetime completo, tenta extrair DD/MM de outras strings
    match = re.search(r'(\d{2}/\d{2})', texto)
    if match:
        return match.group(1)

    return texto # Retorna original se nenhum padrão for encontrado

linhas_metricas = []

for _, row in DF_MERGED.iterrows():
    pagina = int(row[COLUNA_CHAVE])
    for campo in CAMPOS_COMUNS:
        col_ocr = f'{campo}_ocr' if f'{campo}_ocr' in row.index else campo
        col_gt  = f'{campo}_gt'  if f'{campo}_gt'  in row.index else campo

        gt  = str(row[col_gt])  if not pd.isna(row.get(col_gt,  '')) else ''
        hyp = str(row[col_ocr]) if not pd.isna(row.get(col_ocr, '')) else ''

        # Aplica normalização específica para campos de data
        if campo in DATE_FIELDS:
            gt_normalized = normalizar_data(gt)
            hyp_normalized = normalizar_data(hyp)
        else:
            gt_normalized = gt
            hyp_normalized = hyp

        linhas_metricas.append({
            'pagina':       pagina,
            'campo':        campo,
            'gt':           gt,
            'ocr':          hyp,
            'Acuracia_Exata': calc_acuracia_exata(gt_normalized, hyp_normalized),
        })

DF_MET = pd.DataFrame(linhas_metricas)
print(f'✅ {len(DF_MET)} comparações calculadas.')

Páginas com correspondência: 164
✅ 1312 comparações calculadas.


In [ ]:
import time

t_inicio = time.time()

# ── Resumo por campo ──────────────────────────────────────────────────────
resumo_campo = DF_MET.groupby('campo').agg(
        Acuracia_Exata   = ('Acuracia_Exata', 'mean'),
    Acertos          = ('Acuracia_Exata', 'sum'),
    n_paginas        = ('pagina',         'count'),
)
resumo_campo['Acuracia_pct']   = (resumo_campo['Acuracia_Exata'] * 100).round(1)
resumo_campo['Acertos']        = resumo_campo['Acertos'].astype(int)

t_fim = time.time()
tempo_execucao = round(t_fim - t_inicio, 4)

# ── Exibição ──────────────────────────────────────────────────────────────
print('RESUMO POR CAMPO')
print('─' * 95)
print(f'  {"Campo":25s}'
      f' {"Acerto%":>8} {"Acertos":>8} {"N":>4}')
print('  ' + '─' * 90)
for campo, row in resumo_campo.iterrows():
    print(f'  {campo:25s} '
                    f'{row["Acuracia_pct"]:>7.1f}% '
          f'{row["Acertos"]:>5}/{int(row["n_paginas"])}')

# ── Resumo por página ─────────────────────────────────────────────────────
resumo_pag = DF_MET.groupby('pagina').agg(
       Acuracia_Exata  = ('Acuracia_Exata', 'mean'),
).round(4)

print('\nRESUMO POR PÁGINA')
print('─' * 65)
print(resumo_pag.to_string())

# ── Tempo de execução ─────────────────────────────────────────────────────
print(f'\nTEMPO DE EXECUÇÃO')
print('─' * 50)
print(f'  Cálculo das métricas : {tempo_execucao:.4f}s')
print(f'  Páginas avaliadas    : {DF_MET["pagina"].nunique()}')
print(f'  Comparações totais   : {len(DF_MET)} '
      f'({DF_MET["pagina"].nunique()} páginas × {DF_MET["campo"].nunique()} campos)')

RESUMO POR CAMPO
───────────────────────────────────────────────────────────────────────────────────────────────
  Campo                      Acerto%  Acertos    N
  ──────────────────────────────────────────────────────────────────────────────────────────
  codigo_inep                  97.0% 159.0/164
  data_fim_obs                 98.8% 162.0/164
  data_inicio_obs              98.8% 162.0/164
  nome_escola                  61.0% 100.0/164
  nome_gre                     87.8% 144.0/164
  pagina                      100.0% 164.0/164
  produto                      99.4% 163.0/164
  total_kg                     99.4% 163.0/164

RESUMO POR PÁGINA
─────────────────────────────────────────────────────────────────
        Acuracia_Exata
pagina                
1                1.000
2                1.000
3                0.875
4                1.000
5                0.875
6                0.875
7                0.875
8                0.875
9                0.875
10               0.875
11    

# PASSO 12 — Gerar relatório de métricas

In [ ]:
std_metricas = {
    "Acuracia_Exata_desvio_padrao": DF_MET['Acuracia_Exata'].std()
}

acuracia_por_campo = []
for campo, row in resumo_campo.iterrows():
    acuracia_por_campo.append({
        "campo": campo,
        "acuracia_pct": row["Acuracia_pct"],
        "acertos": int(row["Acertos"]),
        "total_comparacoes": int(row["n_paginas"])
    })

media_geral_metricas_completa = {
    "nome_ferramenta": NOME_FERRAMENTA,
    "metricas_gerais": {
        "Acuracia_Exata_media": DF_MET['Acuracia_Exata'].mean(),
        **std_metricas  # Add standard deviations
    },
    "acuracia_por_campo": acuracia_por_campo
}

nome_arquivo_metricas = f"medida_{NOME_FERRAMENTA}.json"

with open(nome_arquivo_metricas, "w", encoding="utf-8") as f:
    json.dump(media_geral_metricas_completa, f, ensure_ascii=False, indent=4)

print(f"\nMédia geral das métricas salva em: {nome_arquivo_metricas}")
print(json.dumps(media_geral_metricas_completa, ensure_ascii=False, indent=4))

files.download(nome_arquivo_metricas)


Média geral das métricas salva em: medida_pymupdf_tesseract.json
{
    "nome_ferramenta": "pymupdf_tesseract",
    "metricas_gerais": {
        "Acuracia_Exata_media": 0.9275914634146342,
        "Acuracia_Exata_desvio_padrao": 0.25926197698699943
    },
    "acuracia_por_campo": [
        {
            "campo": "codigo_inep",
            "acuracia_pct": 97.0,
            "acertos": 159,
            "total_comparacoes": 164
        },
        {
            "campo": "data_fim_obs",
            "acuracia_pct": 98.8,
            "acertos": 162,
            "total_comparacoes": 164
        },
        {
            "campo": "data_inicio_obs",
            "acuracia_pct": 98.8,
            "acertos": 162,
            "total_comparacoes": 164
        },
        {
            "campo": "nome_escola",
            "acuracia_pct": 61.0,
            "acertos": 100,
            "total_comparacoes": 164
        },
        {
            "campo": "nome_gre",
            "acuracia_pct": 87.8,
          

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>